[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/10_snow-data/10_01_downloading_snow_data.ipynb)

# Module 10, Lesson 1: Downloading and Preparing Snow Data (UA 800 m and SNODAS 1 km)
## Automating live downloads, inspecting file formats, and packaging watershed GeoTIFF stacks for the Minnesota River Basin

In this introductory lesson, you connect directly to three public water and snow data services and prepare clean gridded files for the **Minnesota River Basin (HUC 4 `0702`, 44,050.3 km²)** and its **12 HUC-8 subbasins** (`07020001` to `07020012`). You do not need to upload any course files beforehand when running in Google Colab. All downloads pull live from USGS, the University of Arizona (UA), and the National Snow and Ice Data Center (NSIDC), with automatic fallback to local `./data/` files when running offline.

You will inspect raw `.nc` (NetCDF) and `.tar` / `.dat.gz` (SNODAS binary) files, convert a watershed subset into compressed GeoTIFF stacks (`ua_800m_*.tif` and `snodas_*.tif`), verify pixel alignment, and save everything in folders that Lessons 2 and 3 can read directly.

### Module 10 at a Glance

| Lesson | Notebook | Focus in Minnesota River HUC 4 (`0702`) |
|---|---|---|
| 1 (this notebook) | `10_01_downloading_snow_data.ipynb` | Live USGS WBD, UA 800 m NetCDF, and SNODAS raw binary `.tar` downloads, format inspection, GeoTIFF stack creation |
| 2 | `10_02_ua_snow_analysis_mapping.ipynb` | UA 800 m `EPSG:5070` reprojection, resampling tests, 196-day HUC-4 and 12-HUC-8 SWE, depth, and volume maps |
| 3 | `10_03_snodas_snow_analysis_mapping.ipynb` | SNODAS 1 km `2013-10-01` grid reconciliation, SWE and `1044` melt runoff analysis, maps, and `SNODAS - UA` comparison |

## Lesson Information

Lesson Level: Intermediate (step-by-step Python code written for readers without an advanced programming background)

Purpose: Teach H&H engineers how to download, inspect, check, and package University of Arizona 800 m and NOAA/NSIDC SNODAS 1 km snow datasets from public servers so Lessons 2 and 3 can analyze and map them without hidden coordinate or format errors.

### Learning Objectives

By the end of this lesson, you will be able to:

1. Query USGS Watershed Boundary Dataset (WBD) web endpoints and save multi-layer watershed boundaries in a GeoPackage (`.gpkg`) file.
2. Scrape University of Arizona Apache directory listings and choose one file per date using release priority (`stable > provisional > early`).
3. Explain what a NetCDF (`.nc`) file stores and read UA `SWE` and `DEPTH` grids using `float64` full-span coordinate math so single-precision `lon`/`lat` rounding never shifts your spatial window.
4. Explain what SNODAS `.tar`, `.dat.gz`, and `.txt.gz` files contain and decode big-endian `int16` (`>i2`) binary grids in plain Python without `osgeo.gdal`.
5. Package clipped daily snow arrays into compressed multi-band GeoTIFF (`.tif`) files with a `0.15°` geographic buffer for Lessons 2 and 3.

### Prerequisites

- Modules 1 to 6 (basic Python variables, loops, `pandas`, `matplotlib`)
- Familiarity with Modules 7 to 9 geospatial concepts (`geopandas`, `rasterio`, Coordinate Reference Systems)

### Inputs and Outputs

| Role | File | Notes |
|---|---|---|
| Input (online in Colab) | USGS WBD REST, UA `DailyData_800m`, NSIDC `G02158` | Live downloads require zero pre-uploaded files |
| Input (offline fallback) | Local `./data/` files | Used automatically if live servers are unreachable |
| Output (shared study area) | `minnesota_river_wbd_0702.gpkg` | USGS WBD HUC-4 (`0702`) and 12 HUC-8 subbasins |
| Output (sample stacks) | `module10_outputs/ua_800m_mn_river_wy2023_sample_{swe,depth}.tif` | 5-date native `EPSG:4269` (`-999` NoData) sample stacks and manifest CSV |
| Output (sample stacks) | `module10_outputs/snodas_mn_river_wy2023_sample_{swe,melt}.tif` | 3-date native `EPSG:4326` (`-9999` NoData) sample stacks and manifest CSV |
| Output (optional 196-day stacks) | `data/ua_800m_mn_river_wy2023_*.tif` and `data/snodas_mn_river_wy2023_*.tif` | Generated when you set `BUILD_FULL_SEASON = True` |
| Output (figures) | `module10_outputs/fig1_mn_river_study_area.png`, `fig2_downloaded_raw_preview.png` | Two 300 dpi figures formatted in Arial or Liberation Sans |

## Using AI in This Lesson

AI coding assistants can write `requests` retry loops and regex filename parsers quickly, though they often point to UA's older 4 km NSIDC archive (`NSIDC-0719`) or recommend legacy `osgeo.gdal` scripts (`pysnodas`) that fail on Colab Python 3.12 and NumPy 2.x.

Keep three verification habits in this lesson:

1. Print server HTML links and `.txt` headers directly before writing parsing loops so you know the exact filenames and timestamps.
2. Check output `transform` values down to `1e-7°` whenever slicing a CONUS raster, because a tiny spacing rounding error accumulates across 3,300 columns into a full-pixel shift.
3. Log every downloaded filename, release status (`early`, `provisional`, `stable`), and access date in a manifest CSV.

Starter questions to try with your AI assistant:

- "Inspect the `UA_SWE_Depth_800m_v1_YYYYMMDD_<status>.nc` filename pattern. Why can calling `drop_duplicates('date')` without an explicit rank dictionary keep the wrong release tier?"
- "In a CONUS NetCDF file where `lon` is stored as `float32`, why does `(float(lon[-1]) - float(lon[0])) / (len(lon) - 1)` match GDAL's GeoTransform while `lon[1] - lon[0]` drifts by nearly one pixel across 3,300 columns?" 

## Part 1: Overview of UA 800 m and SNODAS 1 km Snow Data

Hydrologic engineers in the United States commonly use two gridded CONUS snow datasets. Understanding their native formats, coordinate systems, NoData flags, and observation hours before writing download scripts prevents silent mismatches in Lessons 2 and 3.

### Side-by-Side Comparison

| Specification | University of Arizona (`DailyData_800m`) | NOAA/NOHRSC SNODAS (`G02158` Masked) |
|---|---|---|
| Method | Assimilates NRCS SNOTEL and NWS COOP snow depth (using Dawson snow density) with 800 m PRISM weather and lidar terrain/forest adjustments | Multi-layer energy and mass balance snow model driven by hourly weather models (`RUC2`, then `RAP`/`HRRR`) and daily airborne gamma, satellite, and ground observations |
| Period of record | WY1982 (`1981-10-01`) to present (~2-day delay) | Sep 30, 2003 to present (routine daily assimilation from Oct 2004) |
| Native file format | NetCDF4 (`.nc`), CF-compliant gridded arrays and metadata | `.tar` archive holding 16 gzipped raw big-endian `int16` (`>i2`) `.dat.gz` grids and `.txt.gz` headers |
| CONUS grid and CRS | `3,105 x 7,025` cells, 30 arc-sec (`0.00833333°`), `EPSG:4269` (NAD83) | `3,351 x 6,935` cells, 30 arc-sec (`0.00833333°`), `EPSG:4326` (WGS84) |
| NoData value | `-999` (lakes, permanent snow/ice, outside CONUS) | `-9999` (oceans and non-US land, plus snow-free land inside product `1044`) |
| Daily time convention | PRISM day ending `12:00 UTC` (`06:00 CST` / `07:00 CDT`) | State snapshot at `06:00 UTC` (`00:00 CST` / `01:00 CDT`), fluxes `06:00 to 06:00 UTC` |
| Data storage | `SWE` and `DEPTH` stored as `int16` in `mm` (`scale = 1`) | Product-specific `int16` scale factors (`/ 1,000` for SWE, `/ 100,000` for Melt) |
| Key items to handle | `early`, `provisional`, and `stable` duplicates, `float32` `lon`/`lat` vectors, no MN SNOTEL stations | `2013-10-01` 5% grid shift, `G02158_missing_files.txt` outages, `2014 to 2019` 126,950 zero-repair cells, `32,767` alpine cap |

### All 8 Archived SNODAS Products (`G02158`)

| Key / File Code | Parameter | Variable Type | Timestamp / Window | Raw Storage to Native Unit (`/ scale_factor`) | To Millimeters (`mm`) |
|---|---|---|---|---|---|
| `1034` (`ssmv11034tS__T0001`) | Snow Water Equivalent (SWE) | State | Snapshot at `06:00 UTC` | `int16 / 1,000` to `m` | `raw_int * 1.0` (`1 raw unit = 1 mm`) |
| `1036` (`ssmv11036tS__T0001`) | Snow Depth | State | Snapshot at `06:00 UTC` | `int16 / 1,000` to `m` | `raw_int * 1.0` (`1 raw unit = 1 mm`) |
| `1038` (`ssmv11038wS__A0024`) | Snow Pack Average Temperature | State | 24-hr avg ending `06:00 UTC` | `int16 / 1` to `K` (integers) | Subtract `273.15` for `°C` (absent before `2004-02-21`) |
| `1044` (`ssmv11044bS__T0024`) | Snow Melt Runoff (base) | Diagnostic | 24-hr total `06:00-06:00 UTC` | `int16 / 100,000` to `m` | `raw_int * 0.01` (`100 raw units = 1 mm`) |
| `1050` (`ssmv11050lL00T0024`) | Sublimation from Snow Pack | Diagnostic | 24-hr total `06:00-06:00 UTC` | `int16 / 100,000` to `m` | `raw_int * 0.01` (`+` condensation, `-` loss) |
| `1039` (`ssmv11039lL00T0024`) | Blowing Snow Sublimation | Diagnostic | 24-hr total `06:00-06:00 UTC` | `int16 / 100,000` to `m` | `raw_int * 0.01` (`<= 0`) |
| `1025` (`ssmv01025SlL01T0024`) | Solid Precipitation (snow) | Driving | 24-hr total `06:00-06:00 UTC` | `int16 / 10` to `kg/m²` | `raw_int * 0.1` (`1 kg/m² = 1 mm` water eq.) |
| `1025` (`ssmv01025SlL00T0024`) | Liquid Precipitation (rain) | Driving | 24-hr total `06:00-06:00 UTC` | `int16 / 10` to `kg/m²` | `raw_int * 0.1` (`1 kg/m² = 1 mm`) |

### Primary Sources and Documentation

- UA 800 m Data Server and Readme: [climate.arizona.edu/data/UA_SWE/DailyData_800m/](https://climate.arizona.edu/data/UA_SWE/DailyData_800m/) and [Readme.txt](https://climate.arizona.edu/data/UA_SWE/Readme.txt)
- UA 4 km NSIDC Archive (`NSIDC-0719`): Broxton, P., X. Zeng, and N. Dawson (2019), [DOI: 10.5067/0GGPB220EX6A](https://doi.org/10.5067/0GGPB220EX6A), [User Guide PDF](https://nsidc.org/sites/default/files/documents/user-guide/nsidc-0719-v001-userguide.pdf), and Broxton et al. (2024), [doi:10.1029/2023EA002964](https://doi.org/10.1029/2023EA002964)
- NOAA / NSIDC SNODAS (`G02158`): [Landing Page](https://nsidc.org/data/g02158/versions/1), [DOI: 10.7265/N5TB14TC](https://doi.org/10.7265/N5TB14TC), [User Guide PDF](https://nsidc.org/sites/default/files/g02158-v001-userguide_2_1.pdf), and [Special Report 11 PDF](https://nsidc.org/sites/default/files/nsidc_special_report_11.pdf)
- SNODAS Archive, Missing-Files Errata, and Zero Repair Mask: [G02158/masked/](https://noaadata.apps.nsidc.org/NOAA/G02158/masked/), [G02158_missing_files.txt](https://noaadata.apps.nsidc.org/NOAA/G02158/G02158_missing_files.txt), and [SNODAS_Zero_Repair_Mask.tif](https://noaadata.apps.nsidc.org/NOAA/G02158/ancillary/SNODAS_Zero_Repair_Mask.tif)
- USGS Watershed Boundary Dataset (WBD): [usgs.gov/national-hydrography/watershed-boundary-dataset](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset)

## Part 2: Workspace Setup and Finding Files in Google Colab

### Installing Required Python Packages

Before importing libraries, the cell below checks whether six geospatial packages are installed in your Python environment and installs any missing ones via `pip`:

- `rasterio` and `rioxarray` read and write georeferenced rasters such as GeoTIFFs (`.tif`) and work with coordinate transforms.
- `geopandas` reads and manipulates vector watershed polygons (`.gpkg` and `.geojson`).
- `xarray`, `netCDF4`, and `cftime` open NetCDF (`.nc`) snow datasets and inspect CF-compliant time coordinates without needing `osgeo.gdal`.

In [ ]:
import importlib.util  # Checks whether a Python library is already installed
import subprocess      # Runs terminal commands such as 'pip install' from Python
import sys             # Gives the path to the currently active Python interpreter

# Dictionary mapping {import_name: pip_package_name}
REQUIRED_PACKAGES = {
    'rasterio': 'rasterio',
    'rioxarray': 'rioxarray',
    'geopandas': 'geopandas',
    'xarray': 'xarray',
    'netCDF4': 'netCDF4',
    'cftime': 'cftime',
}

# Build a list of only the libraries that are missing in this environment
missing = []
for module_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:  # None means not installed
        missing.append(pip_name)

# Install missing libraries using the active Python environment's pip
if missing:
    print(f"Installing missing packages: {', '.join(missing)}")
    # '*missing' unpacks the list into separate command-line arguments, and '-q' keeps pip output short
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Package installation complete.")
else:
    print("All required packages are already installed.")

### Importing Libraries and Defining Map Styling Helpers

The next cell imports standard Python tools for files and archives (`pathlib`, `tarfile`, `gzip`, `zipfile`), web downloads (`requests`), tabular/array math (`pandas`, `numpy`), geospatial boundaries and rasters (`geopandas`, `pyproj`, `rasterio`, `xarray`), and plotting (`matplotlib`). It also sets `Arial` (falling back to `Liberation Sans` in Linux/Colab) and defines small helper functions to draw a scale bar, north arrow, and kilometer axis labels on maps.

In [ ]:
# Standard library tools
import datetime
import gzip
import importlib.util
import io
import math
import re
import shutil
import tarfile
import time
import zipfile
from pathlib import Path

# Arrays, tables, and HTTP requests
import numpy as np
import pandas as pd
import requests

# Vector boundaries and geodesic distances
import geopandas as gpd
from pyproj import CRS, Geod

# Raster GeoTIFF and NetCDF readers
import rasterio
from rasterio.transform import Affine, from_bounds, from_origin
from rasterio.windows import Window, from_bounds as window_from_bounds
import xarray as xr

# Plotting and colormaps
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

# Configure Arial font (Liberation Sans is metrically identical in Colab Linux)
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': FONT_PREFERENCE,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Collect all font family names installed on this machine using a set comprehension
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
active_font = 'DejaVu Sans'  # Safe default if none of the preferred fonts exist
for font_name in FONT_PREFERENCE:
    if font_name in installed_fonts:
        active_font = font_name
        break  # Stop at the highest-priority font found
print(f"Active figure font: {active_font}")

# Custom continuous color palette from near-white snow-free tint to deep purple peak SWE
SNOW_CMAP = LinearSegmentedColormap.from_list(
    'm10_snow', ['#F4F8FB', '#D6EAF8', '#85C1E9', '#2E86C1', '#1B4F72', '#4A235A'], N=256
)
SNOW_CMAP.set_bad('white')  # Paint NaN (NoData or masked) pixels solid white

# Warm amber-to-brown palette for snowmelt runoff rates
MELT_CMAP = LinearSegmentedColormap.from_list(
    'm10_melt', ['#FEF9E7', '#FAD7A0', '#EB984E', '#CA6F1E', '#873600'], N=256
)
MELT_CMAP.set_bad('white')


def add_scalebar(ax, length_km=50.0, location=(0.04, 0.05), show_mid=True):
    """Draw an alternating black and white metric scale bar on an EPSG:5070 map."""
    # Read the current map axis limits in meters (EPSG:5070 units)
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    # Convert fractional placement (e.g., 4% from left, 5% from bottom) into meter coordinates
    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)
    bar_len = length_km * 1000.0        # Convert kilometers to meters
    bar_h = 0.015 * (ymax - ymin)       # Make bar height 1.5% of the map height
    half = bar_len / 2.0

    # Draw left black half and right white half rectangles at drawing layer zorder=10 (on top)
    ax.add_patch(Rectangle((x0, y0), half, bar_h, facecolor='#1C2833', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.add_patch(Rectangle((x0 + half, y0), half, bar_h, facecolor='white', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.text(x0, y0 + bar_h * 1.4, '0', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)
    if show_mid:
        ax.text(x0 + half, y0 + bar_h * 1.4, f'{int(length_km / 2)}', ha='center', va='bottom', fontsize=8, zorder=10)
    ax.text(x0 + bar_len, y0 + bar_h * 1.4, f'{int(length_km)} km', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)


def add_north_arrow(ax, x_frac=0.94, y_frac=0.88):
    """Draw a north arrow using relative axis coordinates (0 to 1)."""
    # xycoords='axes fraction' places the arrow at fixed normalized percentages regardless of data scale
    ax.annotate(
        'N',
        xy=(x_frac, y_frac + 0.07),      # Arrow tip position
        xytext=(x_frac, y_frac),         # Text ('N') anchor position at arrow base
        xycoords='axes fraction',
        textcoords='axes fraction',
        ha='center',
        va='center',
        fontsize=10,
        fontweight='bold',
        color='#1C2833',
        arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#1C2833', mutation_scale=13),
        zorder=10,
    )


def format_km_axes(ax):
    """Convert EPSG:5070 meter coordinates on plot axes into kilometers."""
    # Custom tick formatter divides raw meter coordinates by 1,000 for readable km labels
    fmt = mpl.ticker.FuncFormatter(lambda val, pos: f'{val / 1000:.0f}')
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)
    ax.set_xlabel('Albers Easting (km, EPSG:5070)')
    ax.set_ylabel('Albers Northing (km, EPSG:5070)')


print(
    f"Versions: numpy {np.__version__}, pandas {pd.__version__}, "
    f"geopandas {gpd.__version__}, rasterio {rasterio.__version__}"
)

### Where Files Are Saved in Google Colab and How to Access Them

When you run this notebook in Google Colab, your active working folder is `/content/`. You can browse every folder and file visually by clicking the **Files** (folder) icon on the left sidebar of Colab.

1. Input data files (`minnesota_river_wbd_0702.gpkg`, full-season `.tif` stacks) live in `/content/data/`, or `/content/drive/MyDrive/module10_snow_data/` when `USE_GOOGLE_DRIVE = True`.
2. Outputs created by this notebook (sample GeoTIFF stacks, CSV manifests, `.png` figures, and `.zip` bundles) live in `/content/module10_outputs/`, or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`.
3. Raw downloaded daily files are cached inside `/content/module10_outputs/ua_live_nc/` and `/content/module10_outputs/snodas_live_tars/` so rerunning a cell skips re-downloading files you already fetched.
4. To save any single file to your computer, hover over it in Colab's left **Files** panel, click the three dots, and select **Download** (or run the `.zip` download cell before the Practice section).
5. Because `/content/` is cleared when a Colab runtime disconnects and is separate across different browser tabs, carry files from Lesson 1 to Lessons 2 and 3 either by setting `USE_GOOGLE_DRIVE = True` across all three notebooks or by downloading `lesson1_snow_data.zip` and uploading it when prompted in Lessons 2 and 3.

The cell below defines our dataset URLs, grid constants, shared folders (`DATA_DIR` and `OUTPUT_DIR`), and simple file-finder and download helper functions (`fetch_bytes` retries up to 3 times if a server is momentarily busy).

In [ ]:
# Study area and server settings
TARGET_HUC4 = '0702'                  # Minnesota River Basin (USGS WBD)
TARGET_CRS = 'EPSG:5070'              # NAD83 / Conus Albers Equal Area (meters)
BBOX_PAD_DEG = 0.15                   # ~12-17 km margin kept around the basin before reprojection

UA_BASE_URL = 'https://climate.arizona.edu/data/UA_SWE/DailyData_800m'
UA_NODATA = -999

SNODAS_BASE_URL = 'https://noaadata.apps.nsidc.org/NOAA/G02158'
SNODAS_MASKED_URL = f'{SNODAS_BASE_URL}/masked'
SNODAS_ERRATA_URL = f'{SNODAS_BASE_URL}/G02158_missing_files.txt'
SNODAS_ROWS = 3351
SNODAS_COLS = 6935
SNODAS_NODATA = -9999
SNODAS_SHIFT_DATE = datetime.date(2013, 10, 1)
SNODAS_BOUNDS_PRE_2013 = (-124.73375000000000, 24.94958333333333, -66.94208333333333, 52.87458333333333)
SNODAS_BOUNDS_POST_2013 = (-124.73333333333333, 24.95000000000000, -66.94166666666667, 52.87500000000000)

# Representative WY2023 dates for quick interactive Lesson 1 downloads
SEASON_DATES = {
    'Early accumulation': '2022-12-15',
    'Mid-winter build':   '2023-01-31',
    'WY2023 Peak SWE':    '2023-04-01',
    'Active spring melt': '2023-04-12',
    'Late meltout':       '2023-05-01',
}
LIVE_SNODAS_DATES = [
    '2023-03-12',  # WY2023 SNODAS Peak SWE in Minnesota River HUC 4
    '2023-04-10',  # Active spring melt transition
    '2023-04-12',  # Peak daily snowmelt runoff day
]

# Set USE_GOOGLE_DRIVE = True to share files across Lesson 1, 2, and 3 Colab tabs via MyDrive
USE_GOOGLE_DRIVE = False

# Detect whether this notebook is running inside Google Colab or on a local machine
try:
    IN_COLAB = (
        importlib.util.find_spec('google') is not None
        and importlib.util.find_spec('google.colab') is not None
    )
except ModuleNotFoundError:
    IN_COLAB = False

# Choose input (DATA_DIR) and output (OUTPUT_DIR) folders based on environment
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')  # Ask permission once to connect Google Drive
    DATA_DIR = Path('/content/drive/MyDrive/module10_snow_data')
    OUTPUT_DIR = DATA_DIR / 'outputs'
else:
    DATA_DIR = Path('data')
    OUTPUT_DIR = Path('module10_outputs')

# Create folders if they do not exist yet (parents=True builds nested directories too)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Cache subfolders store raw downloaded .nc and .tar files so reruns avoid re-downloading
LIVE_NC_DIR = OUTPUT_DIR / 'ua_live_nc'
LIVE_NC_DIR.mkdir(parents=True, exist_ok=True)
LIVE_TAR_DIR = OUTPUT_DIR / 'snodas_live_tars'
LIVE_TAR_DIR.mkdir(parents=True, exist_ok=True)


def _normalize_upload_name(raw_name):
    """Remove browser duplicate tags like 'file (1).tif' -> 'file.tif'."""
    p = Path(raw_name)
    # Match optional spaces plus '(digits)' right before the extension and strip them
    stem = re.sub(r'\s*\(\d+\)$', '', p.stem)
    return f"{stem}{p.suffix}"


def _locate_local_file(filename):
    """Look for a file in DATA_DIR, ./data/, /content/, OUTPUT_DIR, and nearby folders."""
    candidates = [
        DATA_DIR / filename,
        Path('data') / filename,
        Path(filename),
        Path('/content/data') / filename,
        Path('/content') / filename,
        OUTPUT_DIR / filename,
        Path('module10_outputs') / filename,
        DATA_DIR / 'outputs' / filename,
    ]
    # Check each expected folder path first and verify the file is non-empty
    for path in candidates:
        if path.exists() and path.is_file() and path.stat().st_size > 0:
            return path
    # Search subfolders inside DATA_DIR recursively in case a zip extracted a nested folder
    if DATA_DIR.exists():
        for match in DATA_DIR.rglob(filename):
            if match.is_file() and match.stat().st_size > 0:
                return match
    # Also check one or two directories below the working folder
    for pattern in (f'*/{filename}', f'*/*/{filename}'):
        for match in Path('.').glob(pattern):
            if match.is_file() and match.stat().st_size > 0:
                return match
    return None


def _save_colab_uploads(uploaded):
    """Save files uploaded in Colab (or extract a .zip archive) into DATA_DIR."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    # colab_files.upload() returns a dict mapping {uploaded_filename: raw_file_bytes}
    for raw_name, content in uploaded.items():
        clean_name = _normalize_upload_name(raw_name)
        raw_cwd_path = Path(raw_name)
        if clean_name.lower().endswith('.zip'):
            # Read the uploaded zip archive directly from memory and unpack each real file into DATA_DIR
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    member_name = Path(info.filename).name
                    # Skip hidden dotfiles and macOS metadata folders inside zip archives
                    if not member_name or member_name.startswith('.') or '__MACOSX' in info.filename:
                        continue
                    dest = DATA_DIR / _normalize_upload_name(member_name)
                    dest.write_bytes(zf.read(info.filename))
                    saved.append(dest.name)
        else:
            dest = DATA_DIR / clean_name
            dest.write_bytes(content)
            saved.append(dest.name)
        # Colab also drops a copy in /content/, so remove the duplicate outside DATA_DIR to keep folders tidy
        if raw_cwd_path.exists() and raw_cwd_path.is_file():
            dest_check = DATA_DIR / clean_name
            if raw_cwd_path.resolve() != dest_check.resolve():
                raw_cwd_path.unlink(missing_ok=True)
    return saved


def find_course_file(filename, allow_upload=True):
    """Return path to a local file, or prompt for upload in Colab if allow_upload is True."""
    hit = _locate_local_file(filename)
    if hit is not None:
        return hit
    # If running in Colab and interactive upload is enabled, prompt the student once
    if IN_COLAB and allow_upload:
        from google.colab import files as colab_files
        print(f"File '{filename}' not found locally. Upload '{filename}' (or a .zip of data/):")
        uploaded = colab_files.upload()
        if uploaded:
            _save_colab_uploads(uploaded)
            hit = _locate_local_file(filename)
            if hit is not None:
                return hit
    raise FileNotFoundError(
        f"Could not find '{filename}' in '{DATA_DIR}', ./data/, or '{OUTPUT_DIR}'."
    )


def fetch_bytes(url, timeout=60, retries=3, session=None):
    """Download bytes from a URL, retrying briefly on temporary network failures."""
    # Reuse a requests.Session when downloading many files, or use requests.get directly
    client = session if session is not None else requests
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.get(url, timeout=timeout)
            resp.raise_for_status()  # Raise an error for HTTP 404/500 responses
            return resp.content
        except requests.RequestException as exc:
            last_err = exc
            # Wait longer between retries (1.5 s, then 3.0 s) before giving up
            if attempt < retries - 1:
                wait_sec = 1.5 * (2 ** attempt)
                time.sleep(wait_sec)
    raise RuntimeError(f"Request failed for {url}: {last_err}")


print(f"Data directory   : {DATA_DIR.resolve()}")
print(f"Outputs directory: {OUTPUT_DIR.resolve()}")

## Part 3: Downloading Watershed Boundaries (USGS WBD HUC 4 `0702`)

Before downloading snow grids across the entire United States, we download the **Minnesota River Basin (`HUC 0702`)** and its **12 HUC-8 subbasins** (`07020001` to `07020012`) so we know the exact geographic rectangle to request.

Two geospatial formats appear in this step:

- **GeoJSON** is a plain-text web format returned by the USGS ArcGIS REST API (`WBD_HUC4_URL` and `WBD_HUC8_URL`). `geopandas.read_file()` reads those GeoJSON bytes directly from memory via `io.BytesIO()`.
- **GeoPackage (`.gpkg`)** is a single SQLite-based GIS file that stores multiple vector layers cleanly together (`layer='huc04'` and `layer='huc08'`).
- A **padded bounding box** adds `0.15°` (~12 to 17 km) to `(west, south, east, north)` so reprojection interpolation kernels in Lessons 2 and 3 keep valid neighbor pixels outside the watershed perimeter.

The cell below queries the live USGS WBD REST service (falling back to `./data/minnesota_river_wbd_0702.gpkg` if offline), verifies that all 12 subbasins loaded (`44,050.3 km²`), and computes padded bounding boxes in both `EPSG:4269` (for UA) and `EPSG:4326` (for SNODAS).

In [ ]:
# USGS ArcGIS REST API queries returning GeoJSON polygons for HUC 4 ('0702') and its HUC-8 subbasins
WBD_HUC4_URL = (
    "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/2/query"
    "?where=huc4%3D%270702%27&outFields=huc4,name,areasqkm,areaacres,states"
    "&outSR=4326&f=geojson"
)
WBD_HUC8_URL = (
    "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query"
    "?where=huc8+LIKE+%270702%25%27&outFields=huc8,name,areasqkm,areaacres,states"
    "&outSR=4326&f=geojson"
)

gpkg_in_data = DATA_DIR / 'minnesota_river_wbd_0702.gpkg'
gpkg_in_out = OUTPUT_DIR / 'minnesota_river_wbd_0702.gpkg'

# Try live USGS download first, falling back to local GeoPackage if running offline
try:
    print("Querying live USGS WBD ArcGIS REST API for HUC 4 0702 and 12 HUC-8 subbasins...")
    huc4_bytes = fetch_bytes(WBD_HUC4_URL, timeout=30, retries=2)
    huc8_bytes = fetch_bytes(WBD_HUC8_URL, timeout=30, retries=2)
    # io.BytesIO wraps raw downloaded bytes so geopandas can read them without writing a temp file
    huc4_geo = gpd.read_file(io.BytesIO(huc4_bytes))
    huc8_geo = gpd.read_file(io.BytesIO(huc8_bytes)).sort_values('huc8').reset_index(drop=True)

    # Repair any minor polygon self-intersections before saving
    if not huc4_geo.is_valid.all():
        huc4_geo['geometry'] = huc4_geo.geometry.buffer(0)
    if not huc8_geo.is_valid.all():
        huc8_geo['geometry'] = huc8_geo.geometry.buffer(0)

    # Save both layers into a single GeoPackage (.gpkg) file in DATA_DIR and OUTPUT_DIR
    if not gpkg_in_data.exists():
        huc4_geo.to_file(gpkg_in_data, layer='huc04', driver='GPKG')
        huc8_geo.to_file(gpkg_in_data, layer='huc08', driver='GPKG')
    huc4_geo.to_file(gpkg_in_out, layer='huc04', driver='GPKG')
    huc8_geo.to_file(gpkg_in_out, layer='huc08', driver='GPKG')
    print(f"Downloaded live WBD boundaries -> saved {gpkg_in_out}")
except Exception as err:
    print(f"Live USGS WBD unavailable ({err}). Loading local GeoPackage fallback...")
    wbd_local = find_course_file('minnesota_river_wbd_0702.gpkg', allow_upload=False)
    huc4_geo = gpd.read_file(wbd_local, layer='huc04')
    huc8_geo = gpd.read_file(wbd_local, layer='huc08').sort_values('huc8').reset_index(drop=True)
    if wbd_local.resolve() != gpkg_in_out.resolve():
        shutil.copy2(wbd_local, gpkg_in_out)  # Copy local file to OUTPUT_DIR preserving metadata
    print(f"Loaded local GeoPackage: {wbd_local}")

# Verify expected watershed counts before continuing
assert len(huc4_geo) == 1 and str(huc4_geo['huc4'].iloc[0]) == TARGET_HUC4
assert len(huc8_geo) == 12, f"Expected 12 HUC-8 subbasins, got {len(huc8_geo)}"

# Reproject polygons to equal-area EPSG:5070 (meters) to calculate true areas
huc4_proj = huc4_geo.to_crs(TARGET_CRS)
huc8_proj = huc8_geo.to_crs(TARGET_CRS)
huc8_proj['short_id'] = huc8_proj['huc8'].str[-2:]              # Last 2 digits ('01' to '12') for map labels
huc8_proj['area_km2'] = huc8_proj.geometry.area / 1e6           # Divide m^2 by 1,000,000 for km^2
huc8_proj['area_sqmi'] = huc8_proj['area_km2'] * 0.386102       # Convert km^2 to square miles
basin_area_km2 = float(huc4_proj.geometry.area.iloc[0] / 1e6)

# Compute 0.15-degree padded bounding boxes (west, south, east, north) for each dataset CRS
west_4269, south_4269, east_4269, north_4269 = [float(v) for v in huc4_geo.to_crs('EPSG:4269').total_bounds]
PADDED_BBOX_4269 = (west_4269 - BBOX_PAD_DEG, south_4269 - BBOX_PAD_DEG, east_4269 + BBOX_PAD_DEG, north_4269 + BBOX_PAD_DEG)

west_4326, south_4326, east_4326, north_4326 = [float(v) for v in huc4_geo.to_crs('EPSG:4326').total_bounds]
PADDED_BBOX_4326 = (west_4326 - BBOX_PAD_DEG, south_4326 - BBOX_PAD_DEG, east_4326 + BBOX_PAD_DEG, north_4326 + BBOX_PAD_DEG)

print(f"\nHUC 4: {huc4_geo['huc4'].iloc[0]} - {huc4_geo['name'].iloc[0]} ({basin_area_km2:,.1f} km², {basin_area_km2 * 0.386102:,.0f} sq mi)")
print(f"Padded EPSG:4269 bbox (UA)    : {[round(v, 3) for v in PADDED_BBOX_4269]}")
print(f"Padded EPSG:4326 bbox (SNODAS): {[round(v, 3) for v in PADDED_BBOX_4326]}")
print("\n12 HUC-8 Subbasins:")
print(huc8_proj[['short_id', 'huc8', 'name', 'states', 'area_km2', 'area_sqmi']].round({'area_km2': 1, 'area_sqmi': 1}).to_string(index=False))

### Visualizing the Downloaded Study Area (Figure 1)

This plotting cell draws the 12 HUC-8 subbasins side-by-side with an area directory table and saves `fig1_mn_river_study_area.png` at 300 dpi inside `OUTPUT_DIR`.

In [ ]:
# Figure 1: Study area map on the left (1.55x width) and subbasin table on the right
fig, (ax_map, ax_tbl) = plt.subplots(
    1, 2, figsize=(11.5, 6.2), gridspec_kw={'width_ratios': [1.55, 1.0]}
)

# Generate 12 distinct pastel colors from Matplotlib's Set3 colormap
palette = plt.cm.Set3(np.linspace(0, 1, 12))
for idx, row in huc8_proj.iterrows():
    # Plot one subbasin polygon at a time using its matching color
    gpd.GeoSeries([row.geometry], crs=TARGET_CRS).plot(
        ax=ax_map, facecolor=palette[idx], edgecolor='#34495E', linewidth=0.9, alpha=0.85
    )
    # representative_point() always lies inside the polygon (unlike centroid on curved shapes)
    pt = row.geometry.representative_point()
    label_x = pt.x
    # Shift label '02' north slightly to avoid crowding neighbor labels
    label_y = pt.y + (18_000 if row['short_id'] == '02' else 0)
    ax_map.text(
        label_x, label_y, row['short_id'],
        ha='center', va='center', fontsize=9, fontweight='bold', color='#17202A',
        bbox=dict(boxstyle='circle,pad=0.20', facecolor='white', edgecolor='#566573', alpha=0.92, lw=0.7)
    )

# Overlay the outer HUC-4 watershed border with a thick line
huc4_proj.boundary.plot(ax=ax_map, color='#0B2545', linewidth=2.2, zorder=5)
ax_map.set_aspect('equal')  # Keep east-west and north-south meter scales identical
x_min, y_min, x_max, y_max = huc4_proj.total_bounds
ax_map.set_xlim(x_min - 20_000, x_max + 20_000)   # Add 20 km margin on left and right
ax_map.set_ylim(y_min - 25_000, y_max + 20_000)   # Add 25 km bottom margin for the scale bar
add_scalebar(ax_map, length_km=50, location=(0.05, 0.05))
add_north_arrow(ax_map, x_frac=0.93, y_frac=0.87)
format_km_axes(ax_map)
ax_map.set_title('Minnesota River Basin (HUC 0702) and 12 HUC-8 Subbasins', pad=10)
ax_map.grid(alpha=0.25, linestyle='--')

# Hide standard axes on the right panel so we can draw a clean text table
ax_tbl.axis('off')
ax_tbl.text(0.0, 0.96, 'Subbasin Directory (EPSG:5070 Area)', fontsize=11, fontweight='bold', va='top')
header_y = 0.90
ax_tbl.text(0.00, header_y, 'ID', fontsize=9, fontweight='bold')
ax_tbl.text(0.09, header_y, 'HUC-8 Name', fontsize=9, fontweight='bold')
ax_tbl.text(0.76, header_y, 'Area (km²)', fontsize=9, fontweight='bold', ha='right')
ax_tbl.text(0.98, header_y, 'States', fontsize=9, fontweight='bold', ha='right')
ax_tbl.plot([0, 1], [0.885, 0.885], color='#1C2833', lw=1.0, transform=ax_tbl.transAxes)

# Draw one color swatch badge and text row per subbasin, stepping downward by 0.068
for idx, row in huc8_proj.iterrows():
    y = 0.835 - idx * 0.068
    ax_tbl.add_patch(Rectangle((0.0, y - 0.015), 0.06, 0.042, facecolor=palette[idx], edgecolor='#34495E', lw=0.6))
    ax_tbl.text(0.03, y + 0.006, row['short_id'], fontsize=8, fontweight='bold', ha='center', va='center')
    ax_tbl.text(0.09, y + 0.006, f"{row['name']} ({row['huc8']})", fontsize=8.5, va='center')
    ax_tbl.text(0.76, y + 0.006, f"{row['area_km2']:,.0f}", fontsize=8.5, va='center', ha='right')
    ax_tbl.text(0.98, y + 0.006, row['states'], fontsize=8.5, va='center', ha='right')

# Bottom summary line and source caption
ax_tbl.plot([0, 1], [0.045, 0.045], color='#1C2833', lw=1.0, transform=ax_tbl.transAxes)
ax_tbl.text(0.09, 0.015, 'Total Minnesota River HUC 4 (0702)', fontsize=9, fontweight='bold')
ax_tbl.text(0.76, 0.015, f"{basin_area_km2:,.0f}", fontsize=9, fontweight='bold', ha='right')
ax_tbl.set_xlim(0, 1)
ax_tbl.set_ylim(0, 1)

fig.subplots_adjust(left=0.07, right=0.98, top=0.92, bottom=0.11, wspace=0.12)
fig.text(
    0.01, 0.01,
    'Boundary source: USGS Watershed Boundary Dataset (WBD) | Map CRS: NAD83 / Conus Albers Equal Area (EPSG:5070)',
    fontsize=8, color='#566573'
)
fig1_path = OUTPUT_DIR / 'fig1_mn_river_study_area.png'
fig.savefig(fig1_path, dpi=300)
plt.show()
print(f"Saved: {fig1_path}")

## Part 4: Downloading and Inspecting University of Arizona 800 m NetCDFs

### Scraping Water-Year Directories and Resolving Release Statuses

The University of Arizona organizes daily files in Water Year folders (`WY1982/` through the current Water Year, where a Water Year runs from **October 1 to September 30**). Each file is named:

`UA_SWE_Depth_800m_v1_YYYYMMDD_<status>.nc`

Because recent files progress through three release tiers (`early` during the current month, `provisional` for 1 to 6 months, and `stable` after 6 months), a Water Year folder can contain more than one file for the same date during transitions.

The cell below defines three straightforward helper functions:

- `water_year_for_date(d)` maps October through December dates into the next calendar year's Water Year (`2022-11-01` belongs to `WY2023`).
- `list_ua_water_year(wy)` downloads the Apache HTML page for `WY{wy}/`, uses `re.findall()` to find every `.nc` link, and stores the date, status, numeric rank (`early=1, provisional=2, stable=3`), filename, and URL in a pandas table.
- `select_latest_status(df)` sorts by date and numeric rank, then drops duplicate dates keeping only the highest-ranked (`last`) file.

In [ ]:
# Numeric maturity ranking used to pick the final file when duplicate dates appear
STATUS_RANK = {'early': 1, 'provisional': 2, 'stable': 3}
# Regular expression capturing (1) 8-digit YYYYMMDD date and (2) release tier name
UA_FILE_RE = re.compile(r'UA_SWE_Depth_800m_v1_(\d{8})_(early|provisional|stable)\.nc$')


def water_year_for_date(date_obj):
    """Return the US Water Year (Oct 1 - Sep 30) for a calendar date."""
    # Oct (month 10), Nov (11), and Dec (12) belong to the following Water Year
    if date_obj.month >= 10:
        return date_obj.year + 1
    return date_obj.year


def list_ua_water_year(wy):
    """Fetch the UA Apache directory page for WY{wy} and return a DataFrame of .nc files."""
    url = f"{UA_BASE_URL}/WY{wy}/"
    # Download the Apache directory HTML listing as text
    html = fetch_bytes(url, timeout=45, retries=2).decode('utf-8', errors='replace')
    # Extract every hyperlink ending in '.nc' from href="..." attributes
    nc_filenames = re.findall(r'href="([^"]+\.nc)"', html, flags=re.IGNORECASE)

    rows = []
    for name in nc_filenames:
        match = UA_FILE_RE.search(name)
        if match:
            date_token = match.group(1)   # Captured 8 digits, e.g. '20230401'
            status = match.group(2)       # Captured tier ('early', 'provisional', or 'stable')
            rows.append({
                'date': pd.to_datetime(date_token, format='%Y%m%d').strftime('%Y-%m-%d'),
                'status': status,
                'rank': STATUS_RANK[status],
                'filename': name,
                'url': f"{url}{name}",
            })
    return pd.DataFrame(rows)


def select_latest_status(df):
    """Keep one row per calendar date, selecting the highest maturity rank."""
    if df.empty:
        return df
    # Sort oldest-to-newest and lowest-to-highest rank so 'last' is always the most mature tier
    sorted_df = df.sort_values(['date', 'rank'])
    dedup_df = sorted_df.drop_duplicates(subset='date', keep='last')
    return dedup_df.sort_values('date').reset_index(drop=True)

### Downloading 5 Sample WY2023 NetCDF Files

Next, we run `list_ua_water_year()` on **WY2023** and the **current active water year**, print how many files exist under each status tier, and download 5 representative WY2023 dates (`SEASON_DATES`) into `LIVE_NC_DIR`. Any file already cached on disk is skipped so reruns finish quickly.

In [ ]:
try:
    # Scrape WY2023 directory HTML and deduplicate any transition-period overlaps
    wy23_raw = list_ua_water_year(2023)
    wy23_clean = select_latest_status(wy23_raw)
    print(
        f"Live WY2023 index: {len(wy23_raw)} files -> {len(wy23_clean)} unique days | "
        f"status counts: {wy23_clean['status'].value_counts().to_dict()}"
    )

    # Check the current active water year (falling back to previous WY if empty)
    cur_wy = water_year_for_date(datetime.date.today())
    active_wy = cur_wy
    active_raw = pd.DataFrame()
    for candidate_wy in (cur_wy, cur_wy - 1):
        try:
            active_raw = list_ua_water_year(candidate_wy)
            if not active_raw.empty:
                active_wy = candidate_wy
                break
        except Exception:
            continue

    if not active_raw.empty:
        active_clean = select_latest_status(active_raw)
        print(
            f"Live WY{active_wy} index: {len(active_raw)} files -> {len(active_clean)} unique days | "
            f"status counts: {active_clean['status'].value_counts().to_dict()}"
        )

    # Download 5 representative WY2023 NetCDF files into LIVE_NC_DIR (skipping already cached files)
    live_nc_paths = {}
    for label, date_str in SEASON_DATES.items():
        match = wy23_clean.loc[wy23_clean['date'] == date_str]
        if match.empty:
            raise RuntimeError(f"Date {date_str} missing from live WY2023 index!")
        fname = match['filename'].iloc[0]
        url = match['url'].iloc[0]
        out_nc = LIVE_NC_DIR / fname

        # Only fetch from the server if the file is missing or zero bytes
        if not out_nc.exists() or out_nc.stat().st_size == 0:
            print(f"  Downloading {label:<20s} ({date_str}) -> {fname}...")
            out_nc.write_bytes(fetch_bytes(url, timeout=90))
        else:
            print(f"  Cached      {label:<20s} ({date_str}) -> {fname}")
        live_nc_paths[date_str] = out_nc

    LIVE_AVAILABLE = True
except Exception as err:
    # Offline mode: subsequent cells will read the local GeoTIFF files in DATA_DIR
    print(f"Live UA server unreachable ({err}). Using local course files if available.")
    live_nc_paths = {}
    LIVE_AVAILABLE = False

### What a NetCDF File (`.nc`) Is and How to Read It Safely

**NetCDF** (Network Common Data Form, extension `.nc`) is a binary file format used across hydrology, meteorology, and climate science. Unlike a flat CSV or image without coordinate labels, a single `.nc` file bundles three things together:

1. **Data arrays** such as `SWE` (`3,105 rows x 7,025 columns`) and `DEPTH`, stored as 16-bit integers (`int16`) in millimeters.
2. **Coordinate vectors** such as `lon` (7,025 longitude centers), `lat` (3,105 latitude centers), and `time` (`days since 1900-01-01`).
3. **Metadata attributes** such as `units`, `_FillValue = -999`, and the `crs` variable (`EPSG:4269`).

One subtle numerical detail governs accuracy when slicing a UA `.nc` file across the United States. UA stores `lon` and `lat` as 32-bit floats (`float32`), which only hold about 7 significant digits. Near `-125°`, subtracting two neighboring longitudes (`lon[1] - lon[0]`) rounds to `0.0083312988°` (a `-2.03e-6°` per-pixel error compared to `0.0083333343°`). Across 3,300 columns eastward to Minnesota (`-97.5°`), that tiny error builds up to `0.81` pixels and shifts the cropped box one full column east (`col_off = 3301`, whereas GDAL starts at `3300`). Every pixel stays the exact same array size `(411, 550)`, yet snow pixels collide with `-999` water masks and create errors up to `1,183 mm`.

Computing grid spacing across the **entire span** in 64-bit floating point (`float64`):

```python
pixel_width_deg = float(abs(lon[-1] - lon[0])) / (len(lon) - 1)
```

eliminates that cumulative drift and matches GDAL's geotransform down to `< 1e-12°`.

The function `open_ua_netcdf_var()` below implements this safe `xarray` reader (with a `rasterio/GDAL` backup if `netCDF4` is absent). Notice we pass `decode_times=False, mask_and_scale=False` so Python keeps raw `int16` millimeters and lets us inspect `_FillValue` and `time` directly.

In [ ]:
def open_ua_netcdf_var(nc_path, var_name, bbox=None):
    """Read SWE or DEPTH from a UA 800 m .nc file and crop to bbox using float64 spacing.

    Returns (array_int16, crs, transform, nodata, metadata_dict).
    """
    meta = {}
    try:
        # Keep raw integers (mask_and_scale=False) and numeric CF time (decode_times=False)
        with xr.open_dataset(nc_path, decode_times=False, mask_and_scale=False) as ds:
            da = ds[var_name]

            # Extract CF time number and units before dropping the single time dimension
            if 'time' in da.dims:
                time_raw = float(ds['time'].values[0])
                time_units = str(ds['time'].attrs.get('units', 'days since 1900-01-01'))
                da = da.isel(time=0)  # Drop the length-1 time axis to get a 2D (lat, lon) grid
            else:
                time_raw = np.nan
                time_units = 'unknown'

            # Ensure array axes are always ordered (lat, lon) so rows=latitude and cols=longitude
            if set(da.dims) == {'lat', 'lon'} and da.dims != ('lat', 'lon'):
                da = da.transpose('lat', 'lon')

            crs_wkt = str(ds['crs'].attrs.get('spatial_ref', 'EPSG:4269')) if 'crs' in ds else 'EPSG:4269'

            # Convert 1D coordinate vectors to float64 and compute spacing across the full span
            lon = ds['lon'].values.astype(np.float64)
            lat = ds['lat'].values.astype(np.float64)
            lon_ascending = bool(lon[-1] > lon[0])
            lat_descending = bool(lat[0] > lat[-1])

            # Divide total span by (N - 1) intervals to prevent float32 rounding drift at col 3300
            pixel_width_deg = float(abs(lon[-1] - lon[0])) / (len(lon) - 1)
            pixel_height_deg = float(abs(lat[-1] - lat[0])) / (len(lat) - 1)
            # NetCDF coordinates mark pixel centers, so shift half a pixel to get the outer top-left corner
            left_lon = float(lon[0] if lon_ascending else lon[-1]) - pixel_width_deg / 2.0
            top_lat = float(lat[0] if lat_descending else lat[-1]) + pixel_height_deg / 2.0
            full_transform = from_origin(left_lon, top_lat, pixel_width_deg, pixel_height_deg)

            raw_nodata = int(da.attrs.get('_FillValue', UA_NODATA))
            meta = {
                'reader': 'xarray/netCDF4',
                'dtype': str(da.encoding.get('dtype', da.dtype)),
                'units': str(da.attrs.get('units', '')),
                'long_name': str(da.attrs.get('long_name', '')),
                'time_raw_value': time_raw,
                'time_units': time_units,
                'full_shape': tuple(da.shape),
            }

            # Orient image top-to-bottom (north at row 0) and west-to-east (west at col 0)
            arr_full = da.values
            if not lat_descending:
                arr_full = arr_full[::-1, :]   # Flip vertically if latitude was stored south-to-north
            if not lon_ascending:
                arr_full = arr_full[:, ::-1]   # Flip horizontally if longitude was stored east-to-west

            if bbox is not None:
                # Convert geographic bbox into integer pixel window indices, rounded outward
                raw_win = window_from_bounds(*bbox, transform=full_transform)
                col_start = max(0, math.floor(raw_win.col_off))
                row_start = max(0, math.floor(raw_win.row_off))
                col_end = min(arr_full.shape[1], math.ceil(raw_win.col_off + raw_win.width))
                row_end = min(arr_full.shape[0], math.ceil(raw_win.row_off + raw_win.height))
                win = Window(col_start, row_start, col_end - col_start, row_end - row_start)
                cropped_arr = np.asarray(arr_full[row_start:row_end, col_start:col_end], dtype=np.int16).copy()
                cropped_tr = rasterio.windows.transform(win, full_transform)
            else:
                cropped_arr = np.asarray(arr_full, dtype=np.int16).copy()
                cropped_tr = full_transform

            return cropped_arr, CRS.from_user_input(crs_wkt), cropped_tr, raw_nodata, meta

    except Exception:
        # Fallback if netCDF4/xarray NetCDF engine is missing: read via GDAL/rasterio netcdf driver
        with rasterio.open(f"netcdf:{nc_path}:{var_name}") as src:
            tags = src.tags()
            btags = src.tags(1)
            meta = {
                'reader': 'rasterio/GDAL',
                'dtype': str(src.dtypes[0]),
                'units': btags.get('units', tags.get(f'{var_name}#units', '')),
                'long_name': btags.get('long_name', tags.get(f'{var_name}#long_name', '')),
                'time_raw_value': float(btags.get('NETCDF_DIM_time', tags.get('NETCDF_DIM_time_VALUES', 'nan'))),
                'time_units': tags.get('time#units', 'days since 1900-01-01'),
                'full_shape': (src.height, src.width),
            }
            if bbox is not None:
                raw_win = window_from_bounds(*bbox, transform=src.transform)
                col_start = max(0, math.floor(raw_win.col_off))
                row_start = max(0, math.floor(raw_win.row_off))
                col_end = min(src.width, math.ceil(raw_win.col_off + raw_win.width))
                row_end = min(src.height, math.ceil(raw_win.row_off + raw_win.height))
                win = Window(col_start, row_start, col_end - col_start, row_end - row_start)
                cropped_arr = src.read(1, window=win).astype(np.int16)
                cropped_tr = src.window_transform(win)
            else:
                cropped_arr = src.read(1).astype(np.int16)
                cropped_tr = src.transform
            nodata_val = int(src.nodata) if src.nodata is not None else UA_NODATA
            return cropped_arr, src.crs, cropped_tr, nodata_val, meta

### Inspecting Peak SWE Metadata and Geographic Cell Widths

Now we call `open_ua_netcdf_var()` on the WY2023 Peak SWE date (`2023-04-01`) and print its array shape, units, `_FillValue`, decoded date, and ground pixel size in meters across southern, middle, and northern Minnesota using `pyproj.Geod`.

In [ ]:
peak_date_str = SEASON_DATES['WY2023 Peak SWE']
# Inspect the live NetCDF when available, or read the matching band from local GeoTIFF
if LIVE_AVAILABLE and peak_date_str in live_nc_paths:
    peak_nc = live_nc_paths[peak_date_str]
    ua_peak_raw, ua_src_crs, ua_src_tr, ua_raw_nodata, nc_meta = open_ua_netcdf_var(peak_nc, 'SWE', PADDED_BBOX_4269)
    print(f"Opened live UA NetCDF via {nc_meta['reader']}: {peak_nc.name}")
    print(f"  Full CONUS shape : {nc_meta['full_shape']} | dtype: {nc_meta['dtype']} | _FillValue: {ua_raw_nodata}")
    print(f"  Units            : {nc_meta['units']} | Variable: {nc_meta['long_name']}")
    print(f"  Subset shape     : {ua_peak_raw.shape} | CRS: {ua_src_crs.to_string()} | pixel deg: ({ua_src_tr.a:.8f}°, {abs(ua_src_tr.e):.8f}°)")
    # Add raw CF days onto base epoch 1900-01-01 to verify calendar date
    t_decoded = pd.Timestamp('1900-01-01') + pd.Timedelta(days=nc_meta['time_raw_value'])
    print(f"  CF time          : {nc_meta['time_raw_value']:.0f} ({nc_meta['time_units']}) -> {t_decoded.strftime('%Y-%m-%d')} (PRISM day ending 12:00 UTC)")
else:
    ua_fallback_tif = find_course_file('ua_800m_mn_river_wy2023_swe.tif', allow_upload=False)
    with rasterio.open(ua_fallback_tif) as src:
        band_number = list(src.descriptions).index(peak_date_str) + 1  # Rasterio bands are 1-indexed
        ua_peak_raw = src.read(band_number)
        ua_src_crs = src.crs
        ua_src_tr = src.transform
        ua_raw_nodata = int(src.nodata)
    print(f"Opened UA Peak SWE from local course GeoTIFF ({ua_fallback_tif.name}): shape={ua_peak_raw.shape}, CRS={ua_src_crs.to_string()}")

# Measure true ground width and height of a 30 arc-sec pixel in EPSG:4269 across HUC 0702
geod = Geod(ellps='GRS80')  # Reference ellipsoid used by NAD83 (EPSG:4269)
dlon = abs(ua_src_tr.a)     # Pixel width in degrees longitude
dlat = abs(ua_src_tr.e)     # Pixel height in degrees latitude
print("\nTrue ground cell dimensions in EPSG:4269 across Minnesota River latitudes:")
lat_checks = [
    ('Southern edge (IA/MN)', south_4269),
    ('Basin midpoint', 0.5 * (south_4269 + north_4269)),
    ('Northern edge (ND/SD/MN)', north_4269),
]
for label, lat_deg in lat_checks:
    # geod.inv returns (forward_azimuth, back_azimuth, surface_distance_in_meters)
    _, _, ew_m = geod.inv(-95.0, lat_deg, -95.0 + dlon, lat_deg)
    _, _, ns_m = geod.inv(-95.0, lat_deg, -95.0, lat_deg + dlat)
    print(f"  {label:<26s} ({lat_deg:.2f}°N): {ew_m:.1f} m E-W x {ns_m:.1f} m N-S (area = {ew_m * ns_m / 1e6:.4f} km²)")

## Part 5: Downloading and Parsing NOAA/NOHRSC SNODAS 1 km Archives

### What SNODAS `.tar`, `.dat.gz`, and `.txt.gz` Files Contain

SNODAS distributes one daily bundle file named `SNODAS_YYYYMMDD.tar`. Inside each `.tar` archive sit **16 gzip-compressed (`.gz`) files** representing 8 snow variables (`1034` SWE, `1044` Snow Melt Runoff, `1036` Snow Depth, `1025` Precipitation, and others):

- `.dat.gz` stores the gridded values as a **flat binary array** with zero header bytes inside the file. Each pixel is stored as a **big-endian signed 16-bit integer** (`>i2` in NumPy, meaning the most significant byte comes first) across `3,351 rows x 6,935 columns`.
- `.txt.gz` stores plain-text `Key: Value` lines documenting units, timestamps, coordinates, and minimum/maximum values.
- `1034` SWE (`ssmv11034...`) is a `06:00 UTC` snapshot stored in `meters / 1,000` (`1 raw unit = 1 mm`).
- `1044` Snow Melt Runoff (`ssmv11044...`) is a 24-hour flux ending `06:00 UTC` stored in `meters / 100,000` (`1 raw unit = 0.01 mm/day`), where snow-free land pixels contain `-9999`.
- Before October 1, 2013 (`2013-10-01`), NOHRSC used slightly shifted grid bounds (`SNODAS_BOUNDS_PRE_2013`). After that date, bounds shifted `+0.00041667°` east and north (`SNODAS_BOUNDS_POST_2013`).

The cell below defines simple pure-Python helpers to pick the right CONUS bounds by date, parse `.txt` lines into a dictionary, and stream `.dat.gz` and `.txt.gz` directly from a `.tar` file using Python's built-in `tarfile` and `gzip` modules.

In [ ]:
# Metadata lookup for the two SNODAS variables extracted in this module
SNODAS_PRODUCT_CODES = {
    '1034': {'name': 'SWE', 'scale_to_m': 1000.0, 'scale_to_mm': 1.0},
    '1044': {'name': 'Snow Melt Runoff (24-hr)', 'scale_to_m': 100000.0, 'scale_to_mm': 0.01},
}


def snodas_bounds_for_date(date_obj):
    """Return (bounds_4326, regime_label) accounting for the Oct 1, 2013 SNODAS grid shift."""
    # Grids before 2013-10-01 sit half a pixel west and south of the modern grid
    if date_obj < SNODAS_SHIFT_DATE:
        return SNODAS_BOUNDS_PRE_2013, 'pre_2013_10_01'
    return SNODAS_BOUNDS_POST_2013, 'post_2013_10_01'


def snodas_full_transform(date_obj):
    """Build the full CONUS Affine geotransform for a given SNODAS date."""
    bounds, _ = snodas_bounds_for_date(date_obj)
    # '*bounds' unpacks (west, south, east, north) into four separate arguments
    return from_bounds(*bounds, SNODAS_COLS, SNODAS_ROWS)


def parse_snodas_txt(raw_text):
    """Parse a SNODAS .txt metadata file ('Key: Value' lines) into a Python dict."""
    parsed = {}
    for line in raw_text.splitlines():
        if ':' in line:
            # split(':', 1) splits only on the first colon so timestamps like 06:00 stay intact
            key, val = line.split(':', 1)
            parsed[key.strip()] = val.strip()
    return parsed


def read_snodas_tar_products(tar_path, date_obj, bbox=None, codes=('1034', '1044')):
    """Extract requested SNODAS binary arrays and text headers from a .tar archive in plain Python."""
    full_tr = snodas_full_transform(date_obj)
    if bbox is not None:
        # Compute integer row/column slice around the padded Minnesota River bbox
        raw_win = window_from_bounds(*bbox, transform=full_tr)
        col_start = max(0, math.floor(raw_win.col_off))
        row_start = max(0, math.floor(raw_win.row_off))
        col_end = min(SNODAS_COLS, math.ceil(raw_win.col_off + raw_win.width))
        row_end = min(SNODAS_ROWS, math.ceil(raw_win.row_off + raw_win.height))
        win = Window(col_start, row_start, col_end - col_start, row_end - row_start)
        out_tr = rasterio.windows.transform(win, full_tr)
    else:
        row_start, row_end, col_start, col_end = 0, SNODAS_ROWS, 0, SNODAS_COLS
        out_tr = full_tr

    arrays = {}
    headers = {}
    filenames = {}
    # 'r:*' lets tarfile read uncompressed or gzipped tar archives automatically
    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf.getmembers():
            mname = Path(member.name).name
            for code in codes:
                if f'ssmv1{code}' in mname:
                    if mname.endswith('.dat.gz'):
                        # Decompress gzipped raw bytes and interpret every 2 bytes as big-endian int16 ('>i2')
                        raw_bytes = gzip.decompress(tf.extractfile(member).read())
                        full_arr = np.frombuffer(raw_bytes, dtype='>i2').reshape(SNODAS_ROWS, SNODAS_COLS)
                        # Slice to the watershed window and copy as native host-endian int16
                        arrays[code] = full_arr[row_start:row_end, col_start:col_end].astype(np.int16, copy=True)
                        filenames[code] = mname
                    elif mname.endswith('.txt.gz') or mname.lower().endswith('.hdr.gz'):
                        txt = gzip.decompress(tf.extractfile(member).read()).decode('latin-1', errors='replace')
                        headers[code] = parse_snodas_txt(txt)
    return arrays, headers, filenames, out_tr

### Checking NSIDC Outages and Downloading Sample SNODAS Tarballs

Next, we read NSIDC's `G02158_missing_files.txt` errata list, download 3 WY2023 `.tar` files (`2023-03-12`, `2023-04-10`, `2023-04-12`) plus the `2013-09-30` / `2013-10-01` grid-shift pair (used in Lesson 3), and print the parsed `1034` SWE and `1044` Melt header metadata.

In [ ]:
# 1. Check NSIDC's missing-files errata text file
try:
    # Normalize Windows line endings and keep only the top Masked-product outage section
    errata_txt = fetch_bytes(SNODAS_ERRATA_URL, timeout=30, retries=2).decode('utf-8', errors='replace').replace('\r\n', '\n')
    masked_block = errata_txt.split('MISSING UNMASKED FILES')[0]
    # Match standalone YYYY-MM-DD lines in that section
    missing_masked_dates = re.findall(r'^\s*(\d{4}-\d{2}-\d{2})\s*$', masked_block, flags=re.MULTILINE)
    # Filter to winter/spring snow months (Nov-Dec and Jan-Apr) using month slice [5:7]
    winter_outages = [d for d in missing_masked_dates if int(d[5:7]) in (11, 12, 1, 2, 3, 4)]
    print(f"Live NSIDC errata: {len(missing_masked_dates)} masked outage days ({missing_masked_dates[0]} to {missing_masked_dates[-1]})")
    print(f"  Nov-Apr outages: {winter_outages}")
except Exception as err:
    print(f"NSIDC errata unreachable ({err}). Documented masked outages: 18 days (2004-2012), including 2007-03-26, 2008-03-13, 2012-12-20.")

# 2. Download 3 WY2023 tar archives and the 2013 grid-shift pair
live_snodas = {}
live_tar_paths = {}
try:
    for date_str in LIVE_SNODAS_DATES:
        date_obj = datetime.date.fromisoformat(date_str)
        tar_name = f"SNODAS_{date_obj:%Y%m%d}.tar"
        tar_path = LIVE_TAR_DIR / tar_name
        if not tar_path.exists() or tar_path.stat().st_size == 0:
            # NSIDC stores tars under /YYYY/MM_Mon/ (for example, /2023/03_Mar/)
            url = f"{SNODAS_MASKED_URL}/{date_obj:%Y}/{date_obj:%m_%b}/{tar_name}"
            print(f"Downloading {tar_name}...")
            tar_path.write_bytes(fetch_bytes(url, timeout=90))
        else:
            print(f"Cached {tar_name}")
        # Extract cropped 1034 (SWE) and 1044 (Melt) arrays and metadata headers
        arrs, hdrs, fnames, tr_sub = read_snodas_tar_products(tar_path, date_obj, PADDED_BBOX_4326)
        live_snodas[date_str] = {'arrays': arrs, 'headers': hdrs, 'filenames': fnames, 'transform': tr_sub}
        live_tar_paths[date_str] = tar_path

    # Download and save 2013-09-30 vs 2013-10-01 subsets for Lesson 3
    shift_rows = []
    for shift_date in (datetime.date(2013, 9, 30), datetime.date(2013, 10, 1)):
        tar_name = f"SNODAS_{shift_date:%Y%m%d}.tar"
        tar_path = LIVE_TAR_DIR / tar_name
        if not tar_path.exists() or tar_path.stat().st_size == 0:
            tar_url = f"{SNODAS_MASKED_URL}/{shift_date:%Y}/{shift_date:%m_%b}/{tar_name}"
            tar_path.write_bytes(fetch_bytes(tar_url, timeout=90))
        s_arrs, s_hdrs, _, s_tr = read_snodas_tar_products(tar_path, shift_date, PADDED_BBOX_4326, codes=('1034',))
        bounds, regime = snodas_bounds_for_date(shift_date)
        sub_fn = f"snodas_{shift_date:%Y%m%d}_swe_subset.tif"

        # Save each shift-pair GeoTIFF in both OUTPUT_DIR and DATA_DIR so Lesson 3 can inspect them
        for target_dir in (OUTPUT_DIR, DATA_DIR):
            out_sub = target_dir / sub_fn
            if target_dir == DATA_DIR and out_sub.exists():
                continue
            with rasterio.open(
                out_sub, 'w', driver='GTiff',
                height=s_arrs['1034'].shape[0], width=s_arrs['1034'].shape[1],
                count=1, dtype='int16', crs='EPSG:4326', transform=s_tr,
                nodata=SNODAS_NODATA, compress='deflate', predictor=2,
            ) as dst:
                dst.write(s_arrs['1034'], 1)
                dst.set_band_description(1, shift_date.isoformat())

        hdr = s_hdrs['1034']
        shift_rows.append({
            'date': shift_date.isoformat(), 'bounds_regime': regime,
            'official_ulx': bounds[0], 'official_uly': bounds[3],
            'official_lrx': bounds[2], 'official_lry': bounds[1],
            'header_benchmark_x': hdr.get('Benchmark x-axis coordinate', ''),
            'header_benchmark_y': hdr.get('Benchmark y-axis coordinate', ''),
            'header_x_offset': hdr.get('X-axis offset', ''),
            'header_y_offset': hdr.get('Y-axis offset', ''),
            'header_min_x': hdr.get('Minimum x-axis coordinate', ''),
            'header_max_x': hdr.get('Maximum x-axis coordinate', ''),
            'header_min_y': hdr.get('Minimum y-axis coordinate', ''),
            'header_max_y': hdr.get('Maximum y-axis coordinate', ''),
            'subset_file': sub_fn,
        })

    # Export side-by-side 2013 header comparison table to CSV
    shift_df = pd.DataFrame(shift_rows)
    shift_df.to_csv(OUTPUT_DIR / 'snodas_2013_shift_headers.csv', index=False)
    if not (DATA_DIR / 'snodas_2013_shift_headers.csv').exists():
        shift_df.to_csv(DATA_DIR / 'snodas_2013_shift_headers.csv', index=False)
    LIVE_SNODAS_OK = True

    pk_d = LIVE_SNODAS_DATES[0]
    h_swe = live_snodas[pk_d]['headers']['1034']
    h_mlt = live_snodas[LIVE_SNODAS_DATES[2]]['headers']['1044']
    print(f"\nParsed .txt header for SWE ({live_snodas[pk_d]['filenames']['1034']}):")
    for key in ['Description', 'Data units', 'Data type', 'No data value',
                'Start year', 'Start month', 'Start day', 'Start hour', 'Stop hour', 'Maximum data value']:
        print(f"  {key:<22s}: {h_swe.get(key, '')}")
    print(
        f"Melt header ({LIVE_SNODAS_DATES[2]}): "
        f"Start {h_mlt['Start year']}-{int(h_mlt['Start month']):02d}-{int(h_mlt['Start day']):02d} {int(h_mlt['Start hour']):02d}:00 UTC"
        f" -> Stop {h_mlt['Stop year']}-{int(h_mlt['Stop month']):02d}-{int(h_mlt['Stop day']):02d} {int(h_mlt['Stop hour']):02d}:00 UTC | Units: {h_mlt['Data units']}"
    )
except Exception as err:
    print(f"Live NSIDC download unavailable ({err}). Using local course files if available.")
    LIVE_SNODAS_OK = False

## Part 6: Packaging Clipped GeoTIFF Stacks and Checking Pixel Accuracy

### What a Multi-Band GeoTIFF Is

A **GeoTIFF (`.tif`)** is a raster image file that embeds map projection (`CRS`), pixel grid spacing and corner coordinates (`transform`), `nodata` flag, and metadata tags inside the image header. By stacking multiple days as numbered **bands** (`Band 1 = Day 1`, `Band 2 = Day 2`, and so on) with lossless `deflate` compression in `int16`, an entire 196-day snow season over a HUC-4 watershed fits into less than 15 MB.

The next cell defines three simple functions using plain `for` loops:

- `write_multiband_geotiff()` saves any 3D NumPy stack `(days, rows, cols)` to a compressed `.tif` with ISO dates stored in each band description.
- `build_ua_geotiff_stack()` loops over a list of dates, downloads any uncached UA `.nc` files via one shared `requests.Session()`, crops `SWE` and `DEPTH` in native `EPSG:4269`, writes both GeoTIFF stacks, and exports a CSV manifest.
- `build_snodas_geotiff_stack()` loops over a list of dates, fetches any uncached SNODAS `.tar` archives, extracts `1034` SWE and `1044` Melt in native `EPSG:4326`, writes both GeoTIFF stacks, and exports a CSV manifest.

In [ ]:
def write_multiband_geotiff(out_path, stack, date_strs, crs, transform, nodata, tags):
    """Save a 3D (bands, rows, cols) int16 array as a lossless DEFLATE-compressed GeoTIFF."""
    # GeoTIFF creation settings (predictor=2 works well for compressing integer grids)
    profile = {
        'driver': 'GTiff',
        'height': stack.shape[1],
        'width': stack.shape[2],
        'count': stack.shape[0],
        'dtype': 'int16',
        'crs': crs,
        'transform': transform,
        'nodata': nodata,
        'compress': 'deflate',
        'predictor': 2,
        'zlevel': 9,
        'tiled': True,
        'blockxsize': min(256, stack.shape[2]),
        'blockysize': min(256, stack.shape[1]),
    }
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(stack)  # Write the full 3D (bands, rows, cols) array at once
        # Label each band with its ISO date ('YYYY-MM-DD') so later lessons can look up bands by date
        for band_num, date_str in enumerate(date_strs, start=1):
            dst.set_band_description(band_num, date_str)
        dst.update_tags(**tags)


def build_ua_geotiff_stack(dates, bbox_4269, out_swe_tif, out_depth_tif, out_manifest_csv, cache_dir=LIVE_NC_DIR):
    """Download UA 800 m .nc files for a list of dates, crop to bbox_4269, and write multi-band GeoTIFFs."""
    # Scrape each needed Water Year index once before starting the daily loop
    unique_wys = sorted({water_year_for_date(d) for d in dates})
    wy_indices = {}
    for wy in unique_wys:
        wy_indices[wy] = select_latest_status(list_ua_water_year(wy))

    swe_slices = []
    depth_slices = []
    manifest_rows = []
    ua_crs = None
    ua_tr = None
    today_iso = datetime.date.today().isoformat()

    # Keep one HTTP session open across all dates to speed up downloads
    with requests.Session() as session:
        total = len(dates)
        for i, date_obj in enumerate(dates, start=1):
            date_str = date_obj.isoformat()
            wy_df = wy_indices[water_year_for_date(date_obj)]
            match = wy_df.loc[wy_df['date'] == date_str]
            if match.empty:
                raise RuntimeError(f"UA 800 m index missing date {date_str}")

            fname = match['filename'].iloc[0]
            status = match['status'].iloc[0]
            url = match['url'].iloc[0]
            nc_path = cache_dir / fname

            # Reuse cached .nc file if already downloaded
            if not nc_path.exists() or nc_path.stat().st_size == 0:
                nc_path.write_bytes(fetch_bytes(url, timeout=90, session=session))

            # Crop both SWE and DEPTH arrays from the daily NetCDF
            swe_arr, ua_crs, ua_tr, _, _ = open_ua_netcdf_var(nc_path, 'SWE', bbox_4269)
            depth_arr, _, _, _, _ = open_ua_netcdf_var(nc_path, 'DEPTH', bbox_4269)
            swe_slices.append(swe_arr)
            depth_slices.append(depth_arr)

            # Record daily QA summary stats for the manifest CSV (excluding -999 NoData)
            valid = swe_arr[swe_arr != UA_NODATA]
            manifest_rows.append({
                'date': date_str,
                'water_year': water_year_for_date(date_obj),
                'filename': fname,
                'status': status,
                'valid_pixels': int(valid.size),
                'min_swe_mm': int(valid.min()) if valid.size else np.nan,
                'max_swe_mm': int(valid.max()) if valid.size else np.nan,
                'access_date': today_iso,
            })
            if total > 10 and (i % 20 == 0 or i == total):
                print(f"  UA progress: {i}/{total}")

    # Stack list of 2D arrays along a new leading axis -> shape (num_days, rows, cols)
    swe_stack = np.stack(swe_slices, axis=0)
    depth_stack = np.stack(depth_slices, axis=0)
    date_strs = [d.isoformat() for d in dates]

    for out_path, stack, var_name in [(out_swe_tif, swe_stack, 'SWE'), (out_depth_tif, depth_stack, 'DEPTH')]:
        write_multiband_geotiff(
            out_path, stack, date_strs, ua_crs, ua_tr, UA_NODATA,
            tags={
                'source': 'University of Arizona 800 m Daily SWE/Depth Dataset',
                'source_url': UA_BASE_URL,
                'nsidc_doi': '10.5067/0GGPB220EX6A',
                'variable': var_name,
                'units': 'mm',
                'scale_factor': '1',
                'time_convention': 'PRISM day ending 12:00 UTC',
                'access_date': today_iso,
                'processing': 'Native EPSG:4269 window slice without reprojection',
            },
        )
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(out_manifest_csv, index=False)
    return manifest_df


def build_snodas_geotiff_stack(dates, bbox_4326, out_swe_tif, out_melt_tif, out_manifest_csv, cache_dir=LIVE_TAR_DIR):
    """Download SNODAS .tar files for a list of dates, crop 1034 SWE and 1044 Melt, and write GeoTIFFs."""
    swe_slices = []
    melt_slices = []
    manifest_rows = []
    sn_tr = None
    today_iso = datetime.date.today().isoformat()

    with requests.Session() as session:
        total = len(dates)
        for i, date_obj in enumerate(dates, start=1):
            tar_name = f"SNODAS_{date_obj:%Y%m%d}.tar"
            tar_path = cache_dir / tar_name
            if not tar_path.exists() or tar_path.stat().st_size == 0:
                url = f"{SNODAS_MASKED_URL}/{date_obj:%Y}/{date_obj:%m_%b}/{tar_name}"
                tar_path.write_bytes(fetch_bytes(url, timeout=90, session=session))

            # Unpack 1034 SWE and 1044 Melt binary grids plus .txt metadata from the tar file
            arrs, hdrs, fnames, sn_tr = read_snodas_tar_products(tar_path, date_obj, bbox_4326, codes=('1034', '1044'))
            swe_arr = arrs['1034']
            melt_arr = arrs['1044']
            swe_slices.append(swe_arr)
            melt_slices.append(melt_arr)

            valid_swe = swe_arr[swe_arr != SNODAS_NODATA]
            valid_melt = melt_arr[melt_arr != SNODAS_NODATA]
            h1034 = hdrs.get('1034', {})
            # Build an ISO UTC timestamp string from parsed .txt header fields
            snap_utc = (
                f"{h1034.get('Start year', date_obj.year)}-{int(h1034.get('Start month', date_obj.month)):02d}-"
                f"{int(h1034.get('Start day', date_obj.day)):02d}T{int(h1034.get('Start hour', 6)):02d}:00:00Z"
            )
            _, regime = snodas_bounds_for_date(date_obj)
            manifest_rows.append({
                'date': date_obj.isoformat(),
                'tar_filename': tar_name,
                'swe_file': fnames.get('1034', ''),
                'melt_file': fnames.get('1044', ''),
                'snapshot_utc': snap_utc,
                'bounds_regime': regime,
                'valid_pixels': int(valid_swe.size),
                'min_swe_mm': int(valid_swe.min()) if valid_swe.size else np.nan,
                'max_swe_mm': int(valid_swe.max()) if valid_swe.size else np.nan,
                'max_melt_mm': round(float(valid_melt.max()) * 0.01, 2) if valid_melt.size else np.nan,
                'access_date': today_iso,
            })
            if total > 10 and (i % 20 == 0 or i == total):
                print(f"  SNODAS progress: {i}/{total}")

    swe_stack = np.stack(swe_slices, axis=0)
    melt_stack = np.stack(melt_slices, axis=0)
    date_strs = [d.isoformat() for d in dates]

    # Save each variable with its own physical units and scale factor in GeoTIFF metadata tags
    specs = [
        (out_swe_tif, swe_stack, '1034', 'SWE', 'meters (raw int16 = mm)', '1000', 'Snapshot at 06:00 UTC'),
        (out_melt_tif, melt_stack, '1044', 'Snow Melt Runoff at Base of Snowpack', 'meters (raw int16 * 0.01 = mm)', '100000', '24-hour total ending 06:00 UTC'),
    ]
    for out_path, stack, code, var_name, units, scale, tdesc in specs:
        write_multiband_geotiff(
            out_path, stack, date_strs, 'EPSG:4326', sn_tr, SNODAS_NODATA,
            tags={
                'source': 'NOAA NOHRSC SNODAS via NSIDC (G02158)',
                'source_url': SNODAS_MASKED_URL,
                'nsidc_doi': '10.7265/N5TB14TC',
                'product_code': code,
                'variable': var_name,
                'units': units,
                'scale_factor': scale,
                'time_convention': tdesc,
                'access_date': today_iso,
                'processing': 'Native EPSG:4326 window slice without reprojection',
            },
        )
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(out_manifest_csv, index=False)
    return manifest_df

### Building Sample GeoTIFF Stacks

The cell below builds our 5-band UA sample GeoTIFFs and 3-band SNODAS sample GeoTIFFs inside `OUTPUT_DIR` and prints their summary manifests. When running offline without server access, it slices the exact same dates from the local course files in `./data/`.

In [ ]:
# Output file paths for the 5-date UA sample and 3-date SNODAS sample stacks
ua_sample_swe_tif = OUTPUT_DIR / 'ua_800m_mn_river_wy2023_sample_swe.tif'
ua_sample_dep_tif = OUTPUT_DIR / 'ua_800m_mn_river_wy2023_sample_depth.tif'
ua_sample_manifest = OUTPUT_DIR / 'ua_800m_mn_river_wy2023_sample_manifest.csv'

sn_sample_swe_tif = OUTPUT_DIR / 'snodas_mn_river_wy2023_sample_swe.tif'
sn_sample_mlt_tif = OUTPUT_DIR / 'snodas_mn_river_wy2023_sample_melt.tif'
sn_sample_manifest = OUTPUT_DIR / 'snodas_mn_river_wy2023_sample_manifest.csv'

# Build UA sample GeoTIFFs from live NetCDFs, or subset local 196-day GeoTIFFs if offline
ua_sample_dates = [datetime.date.fromisoformat(d) for d in SEASON_DATES.values()]
if LIVE_AVAILABLE:
    ua_manifest_df = build_ua_geotiff_stack(
        ua_sample_dates, PADDED_BBOX_4269, ua_sample_swe_tif, ua_sample_dep_tif, ua_sample_manifest
    )
    print(f"Built live UA sample GeoTIFF stacks ({len(ua_sample_dates)} bands) -> {ua_sample_swe_tif.name}")
else:
    swe_full_path = find_course_file('ua_800m_mn_river_wy2023_swe.tif', allow_upload=False)
    dep_full_path = find_course_file('ua_800m_mn_river_wy2023_depth.tif', allow_upload=False)
    date_strs = [d.isoformat() for d in ua_sample_dates]
    rows = []
    with rasterio.open(swe_full_path) as swe_src, rasterio.open(dep_full_path) as depth_src:
        # Look up the 1-based band index corresponding to each selected sample date
        band_nums = [list(swe_src.descriptions).index(ds_str) + 1 for ds_str in date_strs]
        swe_sub = np.stack([swe_src.read(b) for b in band_nums])
        depth_sub = np.stack([depth_src.read(b) for b in band_nums])
        write_multiband_geotiff(ua_sample_swe_tif, swe_sub, date_strs, swe_src.crs, swe_src.transform, UA_NODATA, swe_src.tags())
        write_multiband_geotiff(ua_sample_dep_tif, depth_sub, date_strs, depth_src.crs, depth_src.transform, UA_NODATA, depth_src.tags())
        for i, ds_str in enumerate(date_strs):
            valid = swe_sub[i][swe_sub[i] != UA_NODATA]
            rows.append({'date': ds_str, 'status': 'stable', 'valid_pixels': int(valid.size), 'max_swe_mm': int(valid.max())})
    ua_manifest_df = pd.DataFrame(rows)
    ua_manifest_df.to_csv(ua_sample_manifest, index=False)
    print(f"Created offline UA sample stacks ({len(date_strs)} bands) from local GeoTIFF.")

# Build SNODAS sample GeoTIFFs from live .tar archives, or subset local GeoTIFFs if offline
sn_sample_dates = [datetime.date.fromisoformat(d) for d in LIVE_SNODAS_DATES]
if LIVE_SNODAS_OK:
    sn_manifest_df = build_snodas_geotiff_stack(
        sn_sample_dates, PADDED_BBOX_4326, sn_sample_swe_tif, sn_sample_mlt_tif, sn_sample_manifest
    )
    print(f"Built live SNODAS sample GeoTIFF stacks ({len(sn_sample_dates)} bands) -> {sn_sample_swe_tif.name}")
else:
    sn_swe_full = find_course_file('snodas_mn_river_wy2023_swe.tif', allow_upload=False)
    sn_mlt_full = find_course_file('snodas_mn_river_wy2023_melt.tif', allow_upload=False)
    date_strs = [d.isoformat() for d in sn_sample_dates]
    rows = []
    with rasterio.open(sn_swe_full) as swe_src, rasterio.open(sn_mlt_full) as melt_src:
        band_nums = [list(swe_src.descriptions).index(ds_str) + 1 for ds_str in date_strs]
        swe_sub = np.stack([swe_src.read(b) for b in band_nums])
        melt_sub = np.stack([melt_src.read(b) for b in band_nums])
        write_multiband_geotiff(sn_sample_swe_tif, swe_sub, date_strs, swe_src.crs, swe_src.transform, SNODAS_NODATA, swe_src.tags())
        write_multiband_geotiff(sn_sample_mlt_tif, melt_sub, date_strs, melt_src.crs, melt_src.transform, SNODAS_NODATA, melt_src.tags())
        for i, ds_str in enumerate(date_strs):
            v_swe = swe_sub[i][swe_sub[i] != SNODAS_NODATA]
            v_mlt = melt_sub[i][melt_sub[i] != SNODAS_NODATA]
            rows.append({
                'date': ds_str,
                'valid_pixels': int(v_swe.size),
                'max_swe_mm': int(v_swe.max()),
                'max_melt_mm': round(float(v_mlt.max()) * 0.01, 2) if v_mlt.size else np.nan,
            })
    sn_manifest_df = pd.DataFrame(rows)
    sn_manifest_df.to_csv(sn_sample_manifest, index=False)
    print(f"Created offline SNODAS sample stacks ({len(date_strs)} bands) from local GeoTIFF.")

print("\nUA Sample Manifest:")
print(ua_manifest_df.to_string(index=False))
print("\nSNODAS Sample Manifest:")
print(sn_manifest_df.to_string(index=False))

### Visual Preview and Pixel Accuracy Check (Figure 2)

Before moving to analysis in Lesson 2, we inspect the raw downloaded grids (`fig2_downloaded_raw_preview.png`) to confirm that the `0.15°` geographic buffer surrounds the Minnesota River boundary cleanly, verify CRS and non-negative physical bounds, and compare live downloads against any local full-season stacks (`0 mm` difference).

In [ ]:
# Figure 2: Raw geographic preview showing HUC 0702 inside the 0.15-degree buffer
# Read back the newly created UA Peak SWE band and its map extent (left, right, bottom, top)
with rasterio.open(ua_sample_swe_tif) as ua_src:
    ua_pk_idx = list(ua_src.descriptions).index('2023-04-01') + 1
    ua_pk_raw_arr = ua_src.read(ua_pk_idx)
    ua_ext = [ua_src.bounds.left, ua_src.bounds.right, ua_src.bounds.bottom, ua_src.bounds.top]
    ua_crs_chk = ua_src.crs

# Read back the SNODAS Peak SWE band and Peak Melt band
with rasterio.open(sn_sample_swe_tif) as swe_src, rasterio.open(sn_sample_mlt_tif) as mlt_src:
    sn_pk_idx = list(swe_src.descriptions).index('2023-03-12') + 1
    sn_ml_idx = list(mlt_src.descriptions).index('2023-04-12') + 1
    sn_pk_raw_arr = swe_src.read(sn_pk_idx)
    sn_ml_swe_mask = (swe_src.read(sn_ml_idx) != SNODAS_NODATA)  # Valid US land pixels on 2023-04-12
    sn_ml_raw_arr = mlt_src.read(sn_ml_idx)
    sn_ext = [swe_src.bounds.left, swe_src.bounds.right, swe_src.bounds.bottom, swe_src.bounds.top]
    sn_crs_chk = swe_src.crs

# Convert integer NoData flags (-999 and -9999) into NaN so land outlines render cleanly
ua_vis = np.where(ua_pk_raw_arr == UA_NODATA, np.nan, ua_pk_raw_arr.astype(np.float32))
sn_swe_vis = np.where(sn_pk_raw_arr == SNODAS_NODATA, np.nan, sn_pk_raw_arr.astype(np.float32))
# Product 1044 uses -9999 for snow-free US land, so map those valid land pixels to 0.0 mm/day and scale by 0.01
sn_ml_vis = np.where(
    ~sn_ml_swe_mask,
    np.nan,
    np.where(sn_ml_raw_arr == SNODAS_NODATA, 0.0, sn_ml_raw_arr.astype(np.float32) * 0.01),
)

# Plot three geographic preview panels side by side
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))
h4_4269 = huc4_geo.to_crs('EPSG:4269')
h4_4326 = huc4_geo.to_crs('EPSG:4326')

panels = [
    (axes[0], ua_vis, ua_ext, h4_4269, SNOW_CMAP, 225, 'SWE (mm)', 'UA 800 m Peak SWE (2023-04-01)\nNative EPSG:4269 + 0.15° Buffer'),
    (axes[1], sn_swe_vis, sn_ext, h4_4326, SNOW_CMAP, 225, 'SWE (mm)', 'SNODAS 1 km Peak SWE (2023-03-12)\nNative EPSG:4326 + 0.15° Buffer'),
    (axes[2], sn_ml_vis, sn_ext, h4_4326, MELT_CMAP, 60, 'Melt Runoff (mm/day)', 'SNODAS 1 km Peak Melt (2023-04-12)\nProduct 1044 (scaled x 0.01)'),
]

# Adjust geographic aspect ratio by 1 / cos(latitude) near 44.8 deg N so pixels look square
geo_aspect = 1.0 / np.cos(np.radians(44.8))
for ax, arr, ext, boundary_gdf, cmap, vmax, clabel, title in panels:
    im = ax.imshow(arr, cmap=cmap, vmin=0, vmax=vmax, extent=ext, aspect=geo_aspect)
    boundary_gdf.boundary.plot(ax=ax, color='#0B2545', lw=1.6)
    ax.set_xlabel('Longitude (°W)')
    ax.set_ylabel('Latitude (°N)')
    ax.set_title(title, fontsize=10.5)
    fig.colorbar(im, ax=ax, shrink=0.84, pad=0.03, label=clabel)

fig.suptitle('Module 10 Lesson 1: Raw Downloaded and Buffered Grids Ready for Lessons 2 and 3',
             fontsize=12.5, fontweight='bold', y=1.01)
fig.subplots_adjust(left=0.05, right=0.98, top=0.84, bottom=0.14, wspace=0.28)
fig2_path = OUTPUT_DIR / 'fig2_downloaded_raw_preview.png'
fig.savefig(fig2_path, dpi=300)
plt.show()
print(f"Saved: {fig2_path}")

# Automated QA/QC verification and live vs full-stack 0 mm pixel check when full stacks exist
qa_checks = [
    ('UA sample CRS is EPSG:4269', CRS.from_user_input(ua_crs_chk).to_epsg() == 4269, ua_crs_chk.to_string()),
    ('SNODAS sample CRS is EPSG:4326', CRS.from_user_input(sn_crs_chk).to_epsg() == 4326, sn_crs_chk.to_string()),
    ('UA valid SWE non-negative', bool(np.nanmin(ua_vis) >= 0), f"min={np.nanmin(ua_vis):.1f} mm"),
    ('SNODAS valid Melt non-negative', bool(np.nanmin(sn_ml_vis) >= 0), f"min={np.nanmin(sn_ml_vis):.2f} mm/d"),
]

full_ua_hit = _locate_local_file('ua_800m_mn_river_wy2023_swe.tif')
if LIVE_AVAILABLE and full_ua_hit is not None:
    with rasterio.open(full_ua_hit) as src:
        band_i = list(src.descriptions).index(SEASON_DATES['WY2023 Peak SWE']) + 1
        course_pk = src.read(band_i)
        # Check both affine geotransform values and integer pixel agreement (0 mm error)
        assert np.allclose(tuple(ua_src_tr), tuple(src.transform), atol=1e-7), "UA transform mismatch!"
        max_diff_ua = int(np.max(np.abs(ua_peak_raw.astype(np.int32) - course_pk.astype(np.int32))))
        qa_checks.append(('Live UA Peak vs course GeoTIFF', max_diff_ua == 0, f"max diff = {max_diff_ua} mm"))

full_sn_hit = _locate_local_file('snodas_mn_river_wy2023_swe.tif')
if LIVE_SNODAS_OK and full_sn_hit is not None:
    with rasterio.open(full_sn_hit) as src:
        band_i = list(src.descriptions).index(LIVE_SNODAS_DATES[0]) + 1
        sn_course_pk = src.read(band_i)
        assert np.allclose(tuple(live_snodas[LIVE_SNODAS_DATES[0]]['transform']), tuple(src.transform), atol=1e-7)
        live_arr = live_snodas[LIVE_SNODAS_DATES[0]]['arrays']['1034']
        max_diff_sn = int(np.max(np.abs(live_arr.astype(np.int32) - sn_course_pk.astype(np.int32))))
        qa_checks.append(('Live SNODAS Peak vs course GeoTIFF', max_diff_sn == 0, f"max diff = {max_diff_sn} mm"))

print("\nQA/QC Verification:")
for label, ok, detail in qa_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:<36s} -> {detail}")
    assert ok, f"QA check failed: {label}" 

### Optional: Download and Build the Complete 196-Day Season Stacks

Lessons 2 and 3 analyze all **196 daily bands (`2022-11-01` to `2023-05-15`)** across WY2023.

- If you already have the `data/` folder locally (or uploaded it in Colab), leave `BUILD_FULL_SEASON = False`.
- If you want to generate all 196-day GeoTIFFs directly from USGS, UA, and NSIDC servers inside Colab (or customize `SEASON_START` and `SEASON_END` for another year), change `BUILD_FULL_SEASON = True` and run the cell below (~4 to 6 minutes for 196 days). Set `USE_GOOGLE_DRIVE = True` in Part 2 if you want those files stored permanently in Google Drive for Lessons 2 and 3.

In [ ]:
BUILD_FULL_SEASON = False  # Set True in Colab to download all 196 days live into DATA_DIR
SEASON_START = datetime.date(2022, 11, 1)
SEASON_END = datetime.date(2023, 5, 15)

if BUILD_FULL_SEASON:
    # Build an inclusive daily list from SEASON_START through SEASON_END
    n_days = (SEASON_END - SEASON_START).days + 1
    full_dates = [SEASON_START + datetime.timedelta(days=i) for i in range(n_days)]
    print(f"Building full {n_days}-day season stacks ({SEASON_START} to {SEASON_END}) in {DATA_DIR}...")
    # Download and write both full-season stacks directly into DATA_DIR for Lessons 2 and 3
    build_ua_geotiff_stack(
        full_dates, PADDED_BBOX_4269,
        DATA_DIR / 'ua_800m_mn_river_wy2023_swe.tif',
        DATA_DIR / 'ua_800m_mn_river_wy2023_depth.tif',
        DATA_DIR / 'ua_800m_mn_river_wy2023_manifest.csv',
    )
    build_snodas_geotiff_stack(
        full_dates, PADDED_BBOX_4326,
        DATA_DIR / 'snodas_mn_river_wy2023_swe.tif',
        DATA_DIR / 'snodas_mn_river_wy2023_melt.tif',
        DATA_DIR / 'snodas_mn_river_wy2023_manifest.csv',
    )
    print("Full-season stacks complete and ready for Lessons 2 and 3.")
else:
    print("BUILD_FULL_SEASON is False (sample stacks built above; existing data/ files will be used for Lessons 2 and 3).")

### Optional: Inspect Saved File Locations and Download Lesson 1 Data

Everything generated in Lesson 1 is saved in `OUTPUT_DIR` (and `DATA_DIR`):

- In Google Colab, click the **Files** folder icon on the left sidebar and open `/content/module10_outputs/` (or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`) to browse `minnesota_river_wbd_0702.gpkg`, sample GeoTIFF stacks, CSV manifests, and `.png` figures.
- Run the cell below to inspect every saved file path and size, and bundle the outputs into `lesson1_snow_data.zip`.
- Change `DOWNLOAD_OUTPUTS_ZIP = True` before running if you want Colab to trigger a browser download of `lesson1_snow_data.zip` to your computer.

In [ ]:
DOWNLOAD_OUTPUTS_ZIP = False  # Set True in Google Colab to download lesson1_snow_data.zip

# Collect all outputs created in Lesson 1 and verify none are missing
output_files = [
    gpkg_in_out,
    ua_sample_swe_tif, ua_sample_dep_tif, ua_sample_manifest,
    sn_sample_swe_tif, sn_sample_mlt_tif, sn_sample_manifest,
    fig1_path, fig2_path,
]

print("Lesson 1 saved files:")
for out_f in output_files:
    assert out_f.exists(), f"Missing output: {out_f}"
    size_kb = out_f.stat().st_size / 1024.0  # File size in kilobytes
    print(f"  {out_f.resolve()} ({size_kb:,.1f} KB)")

# Bundle all outputs into a single compressed zip archive inside OUTPUT_DIR
zip_path = OUTPUT_DIR / 'lesson1_snow_data.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for out_f in output_files:
        # arcname=out_f.name stores flat filenames in the zip without deep folder paths
        zf.write(out_f, arcname=out_f.name)
print(f"\nPackaged zip archive: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:,.1f} KB)")

# Trigger a browser download when running inside Google Colab
if DOWNLOAD_OUTPUTS_ZIP:
    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    else:
        print("Running locally: copy or open the .zip path printed above.")

## Part 7: Practice Debugging 1: Duplicate UA Release Statuses

During transition weeks, a UA Water Year folder or local cache can hold two `.nc` files for the same calendar date, one labeled `provisional` and one labeled `stable`. If a download script reads every file in the folder without filtering by `STATUS_RANK`, April 1 to 10 get counted twice and skew monthly averages upward.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 1: THIS CELL CONTAINS AN INTENTIONAL BUG.
# It simulates a folder where April 1-10 exist in BOTH provisional
# and stable versions, then averages April without deduplication.
# ---------------------------------------------------------------
april_dates = pd.date_range('2023-04-01', '2023-04-30', freq='D')
summary_hit = _locate_local_file('ua_800m_mn_river_wy2023_basin_summary.csv')
if summary_hit is not None:
    ref_df = pd.read_csv(summary_hit, parse_dates=['date'])
    april_stable = ref_df[(ref_df['date'] >= '2023-04-01') & (ref_df['date'] <= '2023-04-30')][['date', 'mean_swe_mm']].copy()
else:
    april_swe_vals = np.maximum(0.0, 123.7 * np.exp(-0.14 * np.arange(30))).round(2)
    april_stable = pd.DataFrame({'date': april_dates, 'mean_swe_mm': april_swe_vals})

april_stable['status'] = 'stable'

# Simulate earlier provisional files cached for April 1-10 (+14% higher before delayed COOP QC)
prov_overlap = april_stable.iloc[:10].copy()
prov_overlap['status'] = 'provisional'
prov_overlap['mean_swe_mm'] = (prov_overlap['mean_swe_mm'] * 1.14).round(2)

mixed_folder_df = pd.concat([prov_overlap, april_stable], ignore_index=True)

# Bug: averaging every row without deduplicating by explicit status priority
buggy_april_mean = mixed_folder_df['mean_swe_mm'].mean()
buggy_row_count = len(mixed_folder_df)
true_april_mean = april_stable['mean_swe_mm'].mean()

print(f"Rows processed for 30-day April : {buggy_row_count} (expected 30)")
print(f"Buggy April mean SWE            : {buggy_april_mean:.2f} mm")
print(f"True April mean SWE (stable)    : {true_april_mean:.2f} mm  (bias: {buggy_april_mean - true_april_mean:+.2f} mm)")

### Your Task (1)

1. Identify why 30 calendar days produced 40 rows and explain what goes wrong if you call `.drop_duplicates('date')` without sorting by an explicit numeric rank first.
2. Map `STATUS_RANK = {'early': 1, 'provisional': 2, 'stable': 3}` onto `mixed_folder_df`, sort by `['date', 'rank']`, drop duplicate dates keeping `'last'`, and assert `len(clean_df) == 30`.

Suggested AI prompt:

*"Here is a pandas table of UA SWE files containing both `provisional` and `stable` records for April 1 to 10. Write a deduplication function that ranks `stable > provisional > early`, asserts 30 unique dates, and verifies the April mean matches `true_april_mean`."*

In [ ]:
# YOUR FIX: deduplicate mixed_folder_df using STATUS_RANK = {'early': 1, 'provisional': 2, 'stable': 3}.
#
# Hints:
# 1. Create mixed_folder_df['rank'] = mixed_folder_df['status'].map(STATUS_RANK).
# 2. Sort by ['date', 'rank'] and call .drop_duplicates(subset='date', keep='last').
# 3. Assert len(dedup_df) == 30 and check np.isclose(dedup_df['mean_swe_mm'].mean(), true_april_mean).

## Part 8: Practice Debugging 2: Single-Precision Coordinate Drift and SNODAS Byte Order

This cell demonstrates two binary and coordinate bugs that finish without raising a Python error while scrambling pixel values:

- Bug A calculates pixel spacing from adjacent `float32` longitudes (`lon[1] - lon[0]`). Across 3,300 CONUS columns to Minnesota, rounding shifts the window one column east (`col_off = 3301`), producing a `1,183 mm` collision near `-999` water-mask cells while keeping the identical array shape.
- Bug B reads big-endian SNODAS `>i2` bytes with little-endian `<i2`, swapping byte pairs so ordinary SWE numbers turn into huge positive and negative spikes.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 2: THIS CELL CONTAINS TWO INTENTIONAL BUGS.
# ---------------------------------------------------------------
# Bug A: Estimating NetCDF pixel spacing from lon[1] - lon[0] in float32
conus_lon_f32 = (
    np.float32(-125.02083333 + 1.0 / 240.0)
    + np.arange(7025, dtype=np.float32) * np.float32(1.0 / 120.0)
).astype(np.float32)

dx_buggy = float(abs(conus_lon_f32[1] - conus_lon_f32[0]))
dx_exact = float(abs(float(conus_lon_f32[-1]) - float(conus_lon_f32[0]))) / (len(conus_lon_f32) - 1)

left_lon_exact = float(conus_lon_f32[0]) - dx_exact / 2.0
left_lon_buggy = float(conus_lon_f32[0]) - dx_buggy / 2.0

col0_exact = math.floor((PADDED_BBOX_4269[0] - left_lon_exact) / dx_exact)
col0_buggy = math.floor((PADDED_BBOX_4269[0] - left_lon_buggy) / dx_buggy)

# Simulate the 1-column east shift on the 2023-04-01 Peak SWE grid
shift_collision_mm = int(np.max(np.abs(ua_pk_raw_arr[:, 1:].astype(np.int32) - ua_pk_raw_arr[:, :-1].astype(np.int32))))
print(f"Bug A (float32 lon[1]-lon[0] = {dx_buggy:.10f}° vs exact {dx_exact:.10f}°):")
print(f"  Window col_off shifts from {col0_exact} -> {col0_buggy} (+{col0_buggy - col0_exact} column east)")
print(f"  Same array shape, 1-col shift creates max pixel error = {shift_collision_mm:,} mm")

# Bug B: Decoding SNODAS big-endian (>i2) bytes as little-endian (<i2)
sn_true_sample = sn_pk_raw_arr.astype('>i2')
raw_bytes_big_endian = sn_true_sample.tobytes()
sn_buggy_little_endian = np.frombuffer(raw_bytes_big_endian, dtype='<i2').reshape(sn_true_sample.shape)
print("\nBug B (reading SNODAS >i2 bytes with dtype='<i2'):")
print(f"  True SWE range (big-endian >i2)    : [{sn_true_sample[sn_true_sample != SNODAS_NODATA].min()}, {sn_true_sample.max()}] mm")
print(f"  Corrupted range (little-endian <i2): [{sn_buggy_little_endian.min()}, {sn_buggy_little_endian.max():,}] mm")

### Your Task (2)

1. Explain why `dx_buggy` stayed close near column `1` (`-125°`) yet accumulated nearly a full pixel error by column `3300` (`-97.5°`), and why checking `.shape` alone missed it.
2. Explain why `100 mm` (`0x0064` in big-endian `>i2`) becomes `25,600 mm` (`0x6400`) when decoded with `<i2`.

Suggested AI prompt:

*"My UA NetCDF xarray loader passed a `.shape` check against GDAL's GeoTIFF yet failed an `assert max_diff == 0` check by `1,183 mm`. Explain how `float32` `lon[1] - lon[0]` spacing and `int16` byte endianness (`>i2` vs `<i2`) cause silent raster corruption, and show `float64` full-span and `np.allclose(transform)` checks."*

In [ ]:
# YOUR FIX:
# 1. Verify full-span float64 spacing matches GDAL's GeoTransform (dx ~ 0.0083333343 deg) so col0 == 3300.
# 2. Decode raw_bytes_big_endian using dtype='>i2' and assert zero difference against sn_true_sample.

## Engineering Interpretation and Best Practices

1. Subset each daily raster to a `0.15°`-padded watershed rectangle in native `EPSG:4269` (UA) or `EPSG:4326` (SNODAS) using lossless `int16` DEFLATE GeoTIFFs. Keeping native integers auditable separates data download from projection choices in Lessons 2 and 3.
2. Record release tiers (`early`, `provisional`, `stable`) in manifest CSVs for UA, and log `G02158_missing_files.txt` outages plus `2013-10-01` or `2014 to 2019` repair masks for SNODAS.
3. Verify `transform` arrays with `np.allclose(tuple(tr_a), tuple(tr_b), atol=1e-7)` alongside `.shape` before subtracting grids from two different file readers.

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| `AssertionError: Course copy differs from live NetCDF by 1183 mm` | Computed `dx = lon[1] - lon[0]` on `float32` coordinates | Compute `dx = abs(float(lon[-1]) - float(lon[0])) / (len(lon) - 1)` in `open_ua_netcdf_var()` |
| `ValueError: unable to decode time units` when opening `.nc` | Missing `netCDF4` or `cftime`, or `decode_times=True` | Pass `decode_times=False, mask_and_scale=False` to `xr.open_dataset()` |
| SNODAS grid shows `25,600 mm` SWE or `-3,624` spikes | Read big-endian binary with little-endian `int16` (`<i2`) | Pass `dtype='>i2'` to `np.frombuffer()` |
| `ModuleNotFoundError: No module named 'osgeo'` | Tried legacy `pysnodas` wrapper on Python 3.12 / NumPy 2 | Use Python's `tarfile` + `gzip` + `np.frombuffer(..., dtype='>i2')` |
| Live USGS, UA, or NSIDC request times out | Temporary server load or offline environment | `fetch_bytes()` retries 3 times with backoff, and offline fallback reads local `./data/` |

## Practice Exercises

### Exercise 1: Audit an Historic Flood Year UA Index (`WY2019`)

Call `list_ua_water_year(2019)` and `select_latest_status()` to check how many daily files exist during the 2019 Midwest flood spring, verify whether any date had duplicate release tiers, and print counts by status.

In [ ]:
# EXERCISE 1: Audit UA WY2019 Apache index (or offline manifest)
# Your code here.

### Exercise 2: Add SNODAS Solid and Liquid Precipitation (`1025`) to `read_snodas_tar_products()`

Modify `read_snodas_tar_products()` to extract Liquid Precipitation (`ssmv01025SlL00`, `/ 10` kg/m² = mm) and Solid Precipitation (`ssmv01025SlL01`, `/ 10` kg/m² = mm) from your cached `2023-04-12` tarball (`LIVE_TAR_DIR`), inspect their `.txt` headers, and print the basin maximum rain and snow in millimeters.

In [ ]:
# EXERCISE 2: Extract SNODAS SlL00 (rain) and SlL01 (snow) from cached tar
# Your code here.

### Exercise 3: Download Bounds Estimator for a Single HUC-8 Subbasin

Filter `huc8_geo` to `07020012` (`Lower Minnesota`), compute its `0.15°`-padded `EPSG:4269` and `EPSG:4326` bounding boxes, and calculate how many rows and columns a clipped UA 800 m (`30 arc-sec`) stack contains compared to full CONUS (`3,105 x 7,025`).

In [ ]:
# EXERCISE 3: Estimate clipped grid size for HUC-8 07020012 (Lower Minnesota)
# Your code here.

## Lesson 1 Summary

You queried USGS WBD boundaries (`HUC 0702` and 12 HUC-8 subbasins), inspected and downloaded University of Arizona 800 m NetCDFs and NOAA/NOHRSC SNODAS 1 km binary `.tar` archives, built compressed native-CRS GeoTIFF stacks with a `0.15°` reprojection buffer, and verified the `float64` full-span coordinate fix (`0 mm` drift).

### Next Notebook

Open `10_02_ua_snow_analysis_mapping.ipynb` (Lesson 2) to reproject the UA 800 m stacks onto an equal-area `EPSG:5070` (`800 m`) grid, compare resampling methods during active spring melt, summarize WY2023 snow water volume across all 12 HUC-8 subbasins, and export maps and tables.